# 05 — LLM Pilot Inference Protocol and Execution

## Purpose

This notebook defines and validates the controlled LLM inference protocol for the primary 46-feature pilot experiment.

It compares two information-equivalent input conditions:

1. structured feature values; and
2. deterministic natural-language text.

Both conditions use the same sampled records, feature names, feature values, feature order, task instructions, output schema, model and inference settings. Only the representation of the network-flow record changes.

The notebook is divided into two stages:

1. **Offline protocol validation** — reload and verify the paired inputs, define the common instructions and output schema, construct requests, and perform leakage and pairing checks without calling an LLM.
2. **Controlled inference** — first run a small paired smoke test and, only after it passes, process the complete pilot sample.

The true `Label` and `Attack` values remain outside the model input and are not loaded until the later evaluation stage.

## Inputs and planned outputs

### Inputs produced by Notebook 04

- `data/interim/representations/primary_46/structured.jsonl`
- `data/interim/representations/primary_46/deterministic_text.jsonl`
- `data/interim/representations/primary_46/equivalence_validation.csv`
- `data/interim/representations/primary_46/manifest.json`
- `configs/feature_set_primary_46.json`

### Protocol files produced by this notebook

- `configs/llm_experiment_config.json`
- `configs/llm_output_schema.json`

### Inference artifacts produced only after the protocol passes

- `data/results/llm_pilot/primary_46/raw_responses.jsonl`
- `data/results/llm_pilot/primary_46/parsed_responses.csv`
- `data/results/llm_pilot/primary_46/request_log.jsonl`
- `data/results/llm_pilot/primary_46/run_manifest.json`

The generated inference artifacts will remain outside Git because they may be large, contain provider-specific metadata and be reproducible from the tracked protocol and input-generation notebooks.

## Load the validated paired inputs

The next cells locate the repository root and reload the four artifacts generated by Notebook 04.

This is intentionally performed from disk rather than reusing variables left in memory by another notebook. It checks that Notebook 05 depends only on saved, reproducible artifacts.

At this stage, the private ground-truth file is deliberately not loaded. Therefore, the LLM request-construction code cannot accidentally include the true class label.

In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import math
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    """Find the repository root from either the project or notebooks directory."""
    start = start.resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "notebooks").is_dir()
            and (candidate / "configs").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing README.md, notebooks/ and configs/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

REPRESENTATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "representations"
    / "primary_46"
)

STRUCTURED_PATH = REPRESENTATION_DIR / "structured.jsonl"
TEXT_PATH = REPRESENTATION_DIR / "deterministic_text.jsonl"
VALIDATION_PATH = REPRESENTATION_DIR / "equivalence_validation.csv"
REPRESENTATION_MANIFEST_PATH = REPRESENTATION_DIR / "manifest.json"
FEATURE_SET_PATH = PROJECT_ROOT / "configs" / "feature_set_primary_46.json"

required_input_paths = [
    STRUCTURED_PATH,
    TEXT_PATH,
    VALIDATION_PATH,
    REPRESENTATION_MANIFEST_PATH,
    FEATURE_SET_PATH,
]

missing_input_paths = [
    path.relative_to(PROJECT_ROOT).as_posix()
    for path in required_input_paths
    if not path.exists()
]

if missing_input_paths:
    raise FileNotFoundError(
        "Required inputs are missing:\n- " + "\n- ".join(missing_input_paths)
    )

pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "representation_directory": str(REPRESENTATION_DIR),
        "required_files_found": len(required_input_paths),
        "missing_required_files": len(missing_input_paths),
    },
    name="value",
)

project_root                    /Users/ruiwang/Developer/compsci742-rui-pilot
representation_directory    /Users/ruiwang/Developer/compsci742-rui-pilot/...
required_files_found                                                        5
missing_required_files                                                      0
Name: value, dtype: object

### Reload the saved artifacts

Each JSONL line contains outer experiment metadata and one model input. The outer `sample_id`, feature-set identifier and payload hash are used for experiment management; they are not part of the network-flow record sent to the LLM.

The reload check confirms that:

- both conditions contain 200 records;
- the saved validation table contains 200 rows;
- the manifest also reports 200 records; and
- the primary feature configuration contains 46 ordered fields.

In [2]:
def read_jsonl(path: Path) -> list[dict]:
    """Read a JSON Lines file into a list of dictionaries."""
    records = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()

            if not stripped:
                continue

            try:
                records.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path.name} at line {line_number}."
                ) from exc

    return records


structured_records = read_jsonl(STRUCTURED_PATH)
text_records = read_jsonl(TEXT_PATH)
equivalence_validation = pd.read_csv(VALIDATION_PATH)

with REPRESENTATION_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    representation_manifest = json.load(handle)

with FEATURE_SET_PATH.open("r", encoding="utf-8") as handle:
    primary_feature_set = json.load(handle)


reloaded_input_summary = pd.Series(
    {
        "structured_records": len(structured_records),
        "text_records": len(text_records),
        "validation_rows": len(equivalence_validation),
        "manifest_record_count": representation_manifest["record_count"],
        "primary_feature_count": len(primary_feature_set["ordered_fields"]),
    },
    name="value",
)

reloaded_input_summary

structured_records       200
text_records             200
validation_rows          200
manifest_record_count    200
primary_feature_count     46
Name: value, dtype: int64

## Inspect the reloaded record structure

Before validating the pairs, the saved JSONL structure is inspected explicitly.

This prevents later code from relying on undocumented assumptions about field names or data types. Only outer experiment metadata is displayed here. The full model input is not printed because it is long and has already been validated in Notebook 04.

The inspection checks:

- the outer keys available in each condition;
- the Python data type of each outer value;
- whether both conditions use the same outer record structure; and
- a short preview of each model input.

In [3]:
def describe_outer_record(record: dict) -> pd.DataFrame:
    """
    Summarise the outer structure of one saved experiment record.

    Parameters
    ----------
    record:
        One JSON object loaded from a representation JSONL file.

    Returns
    -------
    pandas.DataFrame
        One row per outer field, showing its name, Python type and a
        shortened preview. The function does not alter the record.
    """
    rows = []

    for field_name, field_value in record.items():
        preview = str(field_value).replace("\n", "\\n")

        if len(preview) > 120:
            preview = preview[:117] + "..."

        rows.append(
            {
                "field_name": field_name,
                "python_type": type(field_value).__name__,
                "preview": preview,
            }
        )

    return pd.DataFrame(rows)


if not structured_records or not text_records:
    raise ValueError(
        "The representation files must contain at least one record before "
        "their structure can be inspected."
    )

structured_outer_structure = describe_outer_record(structured_records[0])
text_outer_structure = describe_outer_record(text_records[0])

outer_key_check = pd.Series(
    {
        "structured_outer_keys": list(structured_records[0].keys()),
        "text_outer_keys": list(text_records[0].keys()),
        "outer_keys_match": (
            list(structured_records[0].keys())
            == list(text_records[0].keys())
        ),
    },
    name="value",
)

display(outer_key_check)
display(structured_outer_structure)
display(text_outer_structure)

structured_outer_keys    [sample_id, feature_set_id, canonical_payload_...
text_outer_keys          [sample_id, feature_set_id, canonical_payload_...
outer_keys_match                                                      True
Name: value, dtype: object

,field_name,python_type,preview
0,sample_id,str,pilot_001
1,feature_set_id,str,primary_46
2,canonical_payload_sha256,str,31739eadb0a14b2490f0fc7e29c86935eef2c22151a332...
3,model_input,str,"{""record_type"":""network_flow"",""features"":[{""na..."


,field_name,python_type,preview
0,sample_id,str,pilot_001
1,feature_set_id,str,primary_46
2,canonical_payload_sha256,str,31739eadb0a14b2490f0fc7e29c86935eef2c22151a332...
3,model_input,str,Network-flow record.\nFeature L4_SRC_PORT has ...


## Validate pairing and outer-record integrity

A paired experiment requires more than an equal number of records. Each structured input must correspond to exactly one deterministic-text input representing the same sampled flow.

The following validation checks:

1. every outer record contains the required metadata fields;
2. `sample_id` is unique within each condition;
3. both conditions contain exactly the same set of sample identifiers;
4. sample order is identical across the two saved files;
5. paired records use the same feature-set identifier;
6. paired records contain the same canonical-payload hash; and
7. every `model_input` is a non-empty string.

Matching payload hashes provide evidence that both rendered inputs were generated from the same canonical feature–value list. They do not compare the literal rendered strings, which are expected to differ between the two representation conditions.

In [4]:
REQUIRED_OUTER_FIELDS = {
    "sample_id",
    "feature_set_id",
    "canonical_payload_sha256",
    "model_input",
}


def validate_outer_record(
    record: dict,
    *,
    condition: str,
    record_number: int,
) -> None:
    """
    Validate the required outer structure of one representation record.

    Parameters
    ----------
    record:
        One JSON object loaded from a representation JSONL file.
    condition:
        Human-readable condition name used in error messages.
    record_number:
        One-based record position used in error messages.

    Raises
    ------
    TypeError
        If the loaded item is not a dictionary.
    ValueError
        If required fields are missing or contain invalid values.
    """
    if not isinstance(record, dict):
        raise TypeError(
            f"{condition} record {record_number} must be a dictionary, "
            f"but found {type(record).__name__}."
        )

    missing_fields = REQUIRED_OUTER_FIELDS.difference(record)

    if missing_fields:
        raise ValueError(
            f"{condition} record {record_number} is missing required fields: "
            f"{sorted(missing_fields)}."
        )

    for metadata_field in (
        "sample_id",
        "feature_set_id",
        "canonical_payload_sha256",
    ):
        value = record[metadata_field]

        if not isinstance(value, str) or not value.strip():
            raise ValueError(
                f"{condition} record {record_number} has an invalid "
                f"{metadata_field!r} value."
            )

    model_input = record["model_input"]

    if not isinstance(model_input, str) or not model_input.strip():
        raise ValueError(
            f"{condition} record {record_number} must contain a non-empty "
            "string in 'model_input'."
        )


for position, record in enumerate(structured_records, start=1):
    validate_outer_record(
        record,
        condition="structured",
        record_number=position,
    )

for position, record in enumerate(text_records, start=1):
    validate_outer_record(
        record,
        condition="deterministic_text",
        record_number=position,
    )


structured_sample_ids = [
    record["sample_id"] for record in structured_records
]
text_sample_ids = [
    record["sample_id"] for record in text_records
]

structured_sample_id_set = set(structured_sample_ids)
text_sample_id_set = set(text_sample_ids)

structured_duplicate_count = (
    len(structured_sample_ids) - len(structured_sample_id_set)
)
text_duplicate_count = (
    len(text_sample_ids) - len(text_sample_id_set)
)

paired_feature_set_matches = [
    structured_record["feature_set_id"]
    == text_record["feature_set_id"]
    for structured_record, text_record in zip(
        structured_records,
        text_records,
        strict=True,
    )
]

paired_payload_hash_matches = [
    structured_record["canonical_payload_sha256"]
    == text_record["canonical_payload_sha256"]
    for structured_record, text_record in zip(
        structured_records,
        text_records,
        strict=True,
    )
]

pairing_validation_summary = pd.Series(
    {
        "structured_record_count": len(structured_records),
        "text_record_count": len(text_records),
        "structured_duplicate_sample_ids": structured_duplicate_count,
        "text_duplicate_sample_ids": text_duplicate_count,
        "sample_id_sets_match": (
            structured_sample_id_set == text_sample_id_set
        ),
        "sample_id_order_matches": (
            structured_sample_ids == text_sample_ids
        ),
        "paired_feature_set_ids_matching": sum(
            paired_feature_set_matches
        ),
        "paired_payload_hashes_matching": sum(
            paired_payload_hash_matches
        ),
        "structured_nonempty_model_inputs": sum(
            bool(record["model_input"].strip())
            for record in structured_records
        ),
        "text_nonempty_model_inputs": sum(
            bool(record["model_input"].strip())
            for record in text_records
        ),
    },
    name="value",
)

pairing_validation_summary

structured_record_count              200
text_record_count                    200
structured_duplicate_sample_ids        0
text_duplicate_sample_ids              0
sample_id_sets_match                True
sample_id_order_matches             True
paired_feature_set_ids_matching      200
paired_payload_hashes_matching       200
structured_nonempty_model_inputs     200
text_nonempty_model_inputs           200
Name: value, dtype: object

### Enforce the pairing requirements

The summary above is intended for human inspection. The following assertions convert the same requirements into execution gates.

If any assertion fails, request construction must stop. This prevents an incomplete, duplicated or incorrectly paired dataset from reaching the LLM inference stage.

In [5]:
expected_record_count = representation_manifest["record_count"]

assert len(structured_records) == expected_record_count, (
    "The structured condition does not match the manifest record count."
)
assert len(text_records) == expected_record_count, (
    "The deterministic-text condition does not match the manifest record count."
)
assert len(equivalence_validation) == expected_record_count, (
    "The equivalence-validation table does not match the manifest record count."
)

assert structured_duplicate_count == 0, (
    "Duplicate sample IDs were found in the structured condition."
)
assert text_duplicate_count == 0, (
    "Duplicate sample IDs were found in the deterministic-text condition."
)
assert structured_sample_id_set == text_sample_id_set, (
    "The two conditions do not contain the same sample-ID set."
)
assert structured_sample_ids == text_sample_ids, (
    "The paired records are not stored in the same sample order."
)
assert all(paired_feature_set_matches), (
    "At least one pair uses different feature-set identifiers."
)
assert all(paired_payload_hash_matches), (
    "At least one pair was generated from a different canonical payload."
)
assert all(
    isinstance(record["model_input"], str)
    and bool(record["model_input"].strip())
    for record in structured_records
), "At least one structured model input is empty or not a string."

assert all(
    isinstance(record["model_input"], str)
    and bool(record["model_input"].strip())
    for record in text_records
), "At least one deterministic-text model input is empty or not a string."

print(
    "Pairing gate passed: "
    f"{expected_record_count} structured/text pairs are complete, unique, "
    "ordered consistently and linked to matching canonical payloads."
)

Pairing gate passed: 200 structured/text pairs are complete, unique, ordered consistently and linked to matching canonical payloads.


## Define the common LLM task protocol

The same task instructions will be used for the structured and deterministic-text conditions. Only the representation enclosed within the record delimiters will change.

### Classification task

The LLM must make one forced binary decision:

- `Benign`: normal, non-attack network traffic; or
- `DoS`: denial-of-service attack traffic associated with attempts to exhaust resources or degrade service availability.

No abstention or third class is permitted because the evaluation dataset and conventional baselines use the same binary task.

### Evidence requirement

The LLM must return exactly five ranked feature citations. For each citation, it must:

1. copy a feature name exactly from the supplied record;
2. copy its observed value exactly from the supplied record; and
3. provide a concise rationale explaining how that supplied feature influenced the prediction.

Exactly five citations are requested so that all records and both representation conditions use the same attribution depth. The order represents the LLM's self-reported importance ranking, from most influential to least influential.

### Information controls

The task instructions do not provide:

- the target record's true label;
- class proportions;
- dataset identity or provenance;
- numerical thresholds;
- normal-value ranges;
- engineered anomaly indicators;
- worked examples; or
- feature-specific interpretations.

These controls reduce opportunities for label leakage, dataset recognition and unequal domain cues.

### Confidence values

A self-reported confidence score is not included in the primary output schema. Such a score is not necessarily a calibrated probability and is not required to answer either research question. Excluding it also reduces output complexity and avoids treating verbal certainty as model probability.

### Interpretation boundary

The cited features and rationales represent the LLM's stated basis for its decision. They must not be interpreted as a complete or causally faithful account of the model's internal reasoning.

### Common instructions

The instructions below are representation-independent. They are stored separately from the network-flow record so that the exact same instruction text can be reused in both conditions.

The record is enclosed within fixed delimiters. These delimiters are identical across conditions and are not part of the underlying flow information.

In [6]:
COMMON_SYSTEM_INSTRUCTION = """You are evaluating a single network-flow record for a controlled binary anomaly-detection experiment.

Classify the record as exactly one of the following:
- Benign: normal, non-attack network traffic.
- DoS: denial-of-service attack traffic associated with attempts to exhaust resources or degrade service availability.

Use only the feature names and observed values supplied in the record. Do not assume access to a hidden label, dataset identity, collection source, class balance, external lookup, normal-value threshold, or additional feature definition.

Return exactly five distinct cited features, ranked from most influential to least influential in your decision. Copy each feature name and observed value exactly as supplied. Give one concise rationale for each cited feature based only on the supplied record.

Return only an object conforming to the required JSON schema. Do not include Markdown, code fences or text outside the JSON object."""


RECORD_OPENING_DELIMITER = "<network_flow_record>"
RECORD_CLOSING_DELIMITER = "</network_flow_record>"


def build_user_message(model_input: str) -> str:
    """
    Enclose one validated representation in the common user-message template.

    The function performs no feature transformation and adds no domain
    interpretation. Therefore, differences between paired user messages are
    limited to the previously defined record representation.

    Parameters
    ----------
    model_input:
        One non-empty structured or deterministic-text representation loaded
        from the validated Notebook 04 artifacts.

    Returns
    -------
    str
        The complete user message to be supplied to the LLM.

    Raises
    ------
    TypeError
        If `model_input` is not a string.
    ValueError
        If `model_input` is empty or already contains a reserved delimiter.
    """
    if not isinstance(model_input, str):
        raise TypeError(
            f"model_input must be a string, not {type(model_input).__name__}."
        )

    if not model_input.strip():
        raise ValueError("model_input must not be empty.")

    reserved_delimiters = (
        RECORD_OPENING_DELIMITER,
        RECORD_CLOSING_DELIMITER,
    )

    if any(delimiter in model_input for delimiter in reserved_delimiters):
        raise ValueError(
            "model_input contains a reserved record delimiter."
        )

    return (
        f"{RECORD_OPENING_DELIMITER}\n"
        f"{model_input}\n"
        f"{RECORD_CLOSING_DELIMITER}"
    )


first_structured_user_message = build_user_message(
    structured_records[0]["model_input"]
)
first_text_user_message = build_user_message(
    text_records[0]["model_input"]
)

pd.Series(
    {
        "system_instruction_characters": len(COMMON_SYSTEM_INSTRUCTION),
        "structured_user_message_characters": len(
            first_structured_user_message
        ),
        "text_user_message_characters": len(
            first_text_user_message
        ),
        "structured_has_opening_delimiter": (
            first_structured_user_message.startswith(
                RECORD_OPENING_DELIMITER
            )
        ),
        "structured_has_closing_delimiter": (
            first_structured_user_message.endswith(
                RECORD_CLOSING_DELIMITER
            )
        ),
        "text_has_opening_delimiter": (
            first_text_user_message.startswith(
                RECORD_OPENING_DELIMITER
            )
        ),
        "text_has_closing_delimiter": (
            first_text_user_message.endswith(
                RECORD_CLOSING_DELIMITER
            )
        ),
    },
    name="value",
)

system_instruction_characters          954
structured_user_message_characters    1914
text_user_message_characters          1891
structured_has_opening_delimiter      True
structured_has_closing_delimiter      True
text_has_opening_delimiter            True
text_has_closing_delimiter            True
Name: value, dtype: object

## Define the response schema

A strict machine-readable schema reduces ambiguity during evaluation.

The response contains:

- one binary prediction; and
- exactly five ranked feature citations.

An overall free-text explanation is not included because it would largely duplicate the five feature rationales and introduce additional unstructured text. The ranked citations themselves provide the explanation material required for the grounding and attribution-agreement analysis.

JSON Schema can enforce the number and structure of citations. A separate post-response validator will later verify that cited feature names are distinct and that both names and observed values occur in the supplied record.

In [7]:
LLM_OUTPUT_SCHEMA = {
    "name": "network_flow_binary_assessment",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "predicted_label": {
                "type": "string",
                "enum": ["Benign", "DoS"],
                "description": (
                    "The forced binary classification for the supplied "
                    "network-flow record."
                ),
            },
            "cited_features": {
                "type": "array",
                "minItems": 5,
                "maxItems": 5,
                "description": (
                    "Exactly five feature citations ranked from most "
                    "influential to least influential."
                ),
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "feature_name": {
                            "type": "string",
                            "description": (
                                "A feature name copied exactly from the "
                                "supplied record."
                            ),
                        },
                        "observed_value": {
                            "type": "string",
                            "description": (
                                "The corresponding value copied exactly "
                                "from the supplied record."
                            ),
                        },
                        "rationale": {
                            "type": "string",
                            "minLength": 1,
                            "description": (
                                "A concise explanation of how this supplied "
                                "feature influenced the prediction."
                            ),
                        },
                    },
                    "required": [
                        "feature_name",
                        "observed_value",
                        "rationale",
                    ],
                },
            },
        },
        "required": [
            "predicted_label",
            "cited_features",
        ],
    },
}


schema_summary = pd.Series(
    {
        "schema_name": LLM_OUTPUT_SCHEMA["name"],
        "strict_mode": LLM_OUTPUT_SCHEMA["strict"],
        "allowed_labels": (
            LLM_OUTPUT_SCHEMA["schema"]["properties"]
            ["predicted_label"]["enum"]
        ),
        "minimum_citations": (
            LLM_OUTPUT_SCHEMA["schema"]["properties"]
            ["cited_features"]["minItems"]
        ),
        "maximum_citations": (
            LLM_OUTPUT_SCHEMA["schema"]["properties"]
            ["cited_features"]["maxItems"]
        ),
        "top_level_additional_properties_allowed": (
            LLM_OUTPUT_SCHEMA["schema"]["additionalProperties"]
        ),
    },
    name="value",
)

schema_summary

schema_name                                network_flow_binary_assessment
strict_mode                                                          True
allowed_labels                                              [Benign, DoS]
minimum_citations                                                       5
maximum_citations                                                       5
top_level_additional_properties_allowed                             False
Name: value, dtype: object

## Parse both representations for grounding validation

Notebook 04 established that the two representations were generated from the same canonical feature–value list. Notebook 05 now defines independent parsers for the saved model inputs.

These parsers are not used to modify the LLM inputs. Their purpose is to construct a validation reference for later response assessment.

For each record, the parsers recover an ordered list of:

- `feature_name`; and
- `observed_value`.

The recovered lists allow the experiment to test whether an LLM:

- cites a feature that was actually supplied;
- copies its observed value correctly;
- cites the same feature more than once; or
- invents an unsupported feature.

The parsers enforce the documented Notebook 04 templates. If an input no longer follows its expected format, validation stops rather than attempting a permissive interpretation.

In [8]:
TEXT_RECORD_HEADER = "Network-flow record."
TEXT_FEATURE_PREFIX = "Feature "
TEXT_NAME_VALUE_SEPARATOR = " has value "
TEXT_FEATURE_SUFFIX = "."

def canonicalise_parsed_json_value(value: object) -> str:
    """
    Convert a parsed JSON scalar to the canonical textual value used for
    cross-representation grounding validation.

    Structured JSON preserves native number types, whereas the deterministic
    text representation expresses every observed value as text. This function
    normalises the parsed JSON value using the same value policy established
    in Notebook 04.

    Parameters
    ----------
    value:
        A JSON scalar recovered from a structured feature value.

    Returns
    -------
    str
        The canonical textual representation used for comparison with the
        deterministic-text condition and later LLM citations.

    Raises
    ------
    TypeError
        If the value is Boolean or is not a supported JSON scalar.
    """
    # Boolean values are rejected explicitly because Python treats bool as a
    # subclass of int, but True/False are not valid quantitative flow values.
    if isinstance(value, bool):
        raise TypeError(
            "Boolean values are not supported as network-flow feature values."
        )

    if value is None:
        return "missing"

    if isinstance(value, str):
        if not value:
            raise ValueError(
                "Structured feature values must not be empty strings."
            )

        return value

    if isinstance(value, int):
        return str(value)

    if isinstance(value, float):
        if math.isnan(value):
            return "missing"

        if math.isinf(value):
            return (
                "positive_infinity"
                if value > 0
                else "negative_infinity"
            )

        # Normalise both positive and negative floating-point zero.
        if value == 0:
            return "0"

        # Avoid adding a meaningless decimal suffix to integer-valued floats.
        if value.is_integer():
            return str(int(value))

        return format(value, ".15g")

    raise TypeError(
        "Unsupported structured feature-value type: "
        f"{type(value).__name__}."
    )


def validate_recovered_features(
    recovered_features: list[dict[str, str]],
    *,
    condition: str,
) -> list[dict[str, str]]:
    """
    Validate a recovered ordered feature–value list.

    Parameters
    ----------
    recovered_features:
        Ordered dictionaries containing `feature_name` and `observed_value`.
    condition:
        Human-readable representation name used in error messages.

    Returns
    -------
    list of dict
        The original ordered list after successful validation.

    Raises
    ------
    ValueError
        If the feature list is empty, contains blank names or values, contains
        duplicate feature names, or does not contain the expected 46 fields.
    """
    expected_feature_count = len(primary_feature_set["ordered_fields"])

    if len(recovered_features) != expected_feature_count:
        raise ValueError(
            f"{condition} input contains {len(recovered_features)} recovered "
            f"features; expected {expected_feature_count}."
        )

    feature_names = []

    for position, feature in enumerate(recovered_features, start=1):
        feature_name = feature.get("feature_name")
        observed_value = feature.get("observed_value")

        if not isinstance(feature_name, str) or not feature_name:
            raise ValueError(
                f"{condition} feature {position} has an invalid feature name."
            )

        if not isinstance(observed_value, str) or not observed_value:
            raise ValueError(
                f"{condition} feature {position} has an invalid observed value."
            )

        feature_names.append(feature_name)

    duplicate_feature_names = sorted(
        {
            feature_name
            for feature_name in feature_names
            if feature_names.count(feature_name) > 1
        }
    )

    if duplicate_feature_names:
        raise ValueError(
            f"{condition} input contains duplicate feature names: "
            f"{duplicate_feature_names}."
        )

    expected_feature_names = primary_feature_set["ordered_fields"]

    if feature_names != expected_feature_names:
        raise ValueError(
            f"{condition} feature names or ordering do not match the locked "
            "primary feature configuration."
        )

    return recovered_features


def parse_structured_model_input(
    model_input: str,
) -> list[dict[str, str]]:
    """
    Recover the ordered features from one structured JSON representation.

    The expected format is a JSON object with `record_type` equal to
    `network_flow` and a `features` array containing only `name` and `value`.

    Parameters
    ----------
    model_input:
        One structured representation generated by Notebook 04.

    Returns
    -------
    list of dict
        Ordered dictionaries using the common validation keys
        `feature_name` and `observed_value`.
    """
    try:
        parsed_record = json.loads(model_input)
    except json.JSONDecodeError as exc:
        raise ValueError(
            "The structured model input is not valid JSON."
        ) from exc

    if not isinstance(parsed_record, dict):
        raise ValueError(
            "The structured model input must decode to a JSON object."
        )

    if set(parsed_record) != {"record_type", "features"}:
        raise ValueError(
            "The structured model input must contain exactly "
            "'record_type' and 'features'."
        )

    if parsed_record["record_type"] != "network_flow":
        raise ValueError(
            "The structured record_type must equal 'network_flow'."
        )

    raw_features = parsed_record["features"]

    if not isinstance(raw_features, list):
        raise ValueError(
            "The structured 'features' field must be a list."
        )

    recovered_features = []

    for position, feature in enumerate(raw_features, start=1):
        if not isinstance(feature, dict):
            raise ValueError(
                f"Structured feature {position} must be a JSON object."
            )

        if set(feature) != {"name", "value"}:
            raise ValueError(
                f"Structured feature {position} must contain exactly "
                "'name' and 'value'."
            )

        if not isinstance(feature["name"], str):
            raise ValueError(
                f"Structured feature {position} has a non-string name."
            )

        if not isinstance(feature["value"], str):
            raise ValueError(
                f"Structured feature {position} has a non-string value."
            )

        recovered_features.append(
            {
                "feature_name": feature["name"],
                "observed_value": feature["value"],
            }
        )

    return validate_recovered_features(
        recovered_features,
        condition="structured",
    )


def parse_text_model_input(
    model_input: str,
) -> list[dict[str, str]]:
    """
    Recover ordered features from one deterministic-text representation.

    The parser accepts only the fixed Notebook 04 template:

    `Network-flow record.`
    `Feature <name> has value <value>.`

    Parameters
    ----------
    model_input:
        One deterministic-text representation generated by Notebook 04.

    Returns
    -------
    list of dict
        Ordered dictionaries using the common validation keys
        `feature_name` and `observed_value`.
    """
    if not isinstance(model_input, str):
        raise TypeError(
            "The deterministic-text model input must be a string."
        )

    lines = model_input.splitlines()

    if not lines or lines[0] != TEXT_RECORD_HEADER:
        raise ValueError(
            "The deterministic-text input has an invalid or missing header."
        )

    recovered_features = []

    for line_number, line in enumerate(lines[1:], start=2):
        if not line.startswith(TEXT_FEATURE_PREFIX):
            raise ValueError(
                f"Text line {line_number} does not begin with "
                f"{TEXT_FEATURE_PREFIX!r}."
            )

        if not line.endswith(TEXT_FEATURE_SUFFIX):
            raise ValueError(
                f"Text line {line_number} does not end with "
                f"{TEXT_FEATURE_SUFFIX!r}."
            )

        feature_body = line[
            len(TEXT_FEATURE_PREFIX) : -len(TEXT_FEATURE_SUFFIX)
        ]

        feature_name, separator, observed_value = feature_body.partition(
            TEXT_NAME_VALUE_SEPARATOR
        )

        if separator != TEXT_NAME_VALUE_SEPARATOR:
            raise ValueError(
                f"Text line {line_number} does not contain the required "
                "name–value separator."
            )

        if not feature_name:
            raise ValueError(
                f"Text line {line_number} contains an empty feature name."
            )

        if not observed_value:
            raise ValueError(
                f"Text line {line_number} contains an empty observed value."
            )

        recovered_features.append(
            {
                "feature_name": feature_name,
                "observed_value": observed_value,
            }
        )

    return validate_recovered_features(
        recovered_features,
        condition="deterministic_text",
    )

### Test the parsers on the first pair

The first structured and deterministic-text inputs are parsed independently. Their recovered feature lists must then match exactly in feature names, values and order.

This is a local parser test. The next section applies the same test to all 200 pairs.

In [9]:
def parse_structured_model_input(
    model_input: str,
) -> list[dict[str, str]]:
    """
    Recover and canonicalise the ordered features from one structured input.

    The structured representation retains native JSON scalar types. Recovered
    values are therefore converted to their canonical textual form before
    comparison with the deterministic-text representation.

    Parameters
    ----------
    model_input:
        One structured JSON representation generated by Notebook 04.

    Returns
    -------
    list of dict
        Ordered dictionaries using the common validation keys
        `feature_name` and `observed_value`.

    Raises
    ------
    TypeError
        If `model_input` is not a string or a feature value uses an unsupported
        type.
    ValueError
        If the JSON or documented structured-input format is invalid.
    """
    if not isinstance(model_input, str):
        raise TypeError(
            "The structured model input must be a string."
        )

    try:
        parsed_record = json.loads(model_input)
    except json.JSONDecodeError as exc:
        raise ValueError(
            "The structured model input is not valid JSON."
        ) from exc

    if not isinstance(parsed_record, dict):
        raise ValueError(
            "The structured model input must decode to a JSON object."
        )

    if set(parsed_record) != {"record_type", "features"}:
        raise ValueError(
            "The structured model input must contain exactly "
            "'record_type' and 'features'."
        )

    if parsed_record["record_type"] != "network_flow":
        raise ValueError(
            "The structured record_type must equal 'network_flow'."
        )

    raw_features = parsed_record["features"]

    if not isinstance(raw_features, list):
        raise ValueError(
            "The structured 'features' field must be a list."
        )

    recovered_features = []

    for position, feature in enumerate(raw_features, start=1):
        if not isinstance(feature, dict):
            raise ValueError(
                f"Structured feature {position} must be a JSON object."
            )

        if set(feature) != {"name", "value"}:
            raise ValueError(
                f"Structured feature {position} must contain exactly "
                "'name' and 'value'."
            )

        feature_name = feature["name"]

        if not isinstance(feature_name, str) or not feature_name:
            raise ValueError(
                f"Structured feature {position} has an invalid name."
            )

        try:
            canonical_value = canonicalise_parsed_json_value(
                feature["value"]
            )
        except (TypeError, ValueError) as exc:
            raise type(exc)(
                f"Structured feature {position} ({feature_name!r}) has an "
                f"invalid value: {exc}"
            ) from exc

        recovered_features.append(
            {
                "feature_name": feature_name,
                "observed_value": canonical_value,
            }
        )

    return validate_recovered_features(
        recovered_features,
        condition="structured",
    )

In [10]:
first_structured_features = parse_structured_model_input(
    structured_records[0]["model_input"]
)
first_text_features = parse_text_model_input(
    text_records[0]["model_input"]
)

first_parser_check = pd.Series(
    {
        "sample_id": structured_records[0]["sample_id"],
        "structured_features_recovered": len(
            first_structured_features
        ),
        "text_features_recovered": len(first_text_features),
        "recovered_lists_match": (
            first_structured_features == first_text_features
        ),
        "first_feature_name": (
            first_structured_features[0]["feature_name"]
        ),
        "first_feature_value": (
            first_structured_features[0]["observed_value"]
        ),
    },
    name="value",
)

display(first_parser_check)
display(pd.DataFrame(first_structured_features).head(10))

sample_id                          pilot_001
structured_features_recovered             46
text_features_recovered                   46
recovered_lists_match                   True
first_feature_name               L4_SRC_PORT
first_feature_value                    47350
Name: value, dtype: object

,feature_name,observed_value
0,L4_SRC_PORT,47350
1,L4_DST_PORT,1581
2,PROTOCOL,6
3,L7_PROTO,0
4,IN_BYTES,492
5,IN_PKTS,10
6,OUT_BYTES,504
7,OUT_PKTS,10
8,TCP_FLAGS,19
9,CLIENT_TCP_FLAGS,19


### Apply the parsers to all paired records

The first-record test confirms that both parsers handle one paired sample correctly. The following cell applies the same independent parsing process to all 200 record pairs.

For each sample, it:

1. parses the structured representation;
2. parses the deterministic-text representation;
3. compares all recovered feature names, values and positions;
4. stops immediately if a paired result differs; and
5. builds a feature–value lookup used later to validate LLM citations.

The lookup contains no true class labels and is not part of the model input.

In [11]:
expected_primary_feature_count = len(
    primary_feature_set["ordered_fields"]
)

grounding_reference_by_sample: dict[str, dict[str, str]] = {}
parser_validation_rows = []

for structured_record, text_record in zip(
    structured_records,
    text_records,
    strict=True,
):
    structured_sample_id = structured_record["sample_id"]
    text_sample_id = text_record["sample_id"]

    if structured_sample_id != text_sample_id:
        raise ValueError(
            "Paired parser validation encountered different sample IDs: "
            f"{structured_sample_id!r} and {text_sample_id!r}."
        )

    structured_features = parse_structured_model_input(
        structured_record["model_input"]
    )
    text_features = parse_text_model_input(
        text_record["model_input"]
    )

    recovered_lists_match = structured_features == text_features

    if not recovered_lists_match:
        raise ValueError(
            "Recovered structured and deterministic-text features differ "
            f"for sample {structured_sample_id!r}."
        )

    # Build one exact feature-name-to-value reference for later grounding
    # validation. Duplicate feature names have already been rejected by the
    # parser, so this dictionary conversion cannot silently overwrite a field.
    grounding_reference_by_sample[structured_sample_id] = {
        feature["feature_name"]: feature["observed_value"]
        for feature in structured_features
    }

    parser_validation_rows.append(
        {
            "sample_id": structured_sample_id,
            "structured_feature_count": len(structured_features),
            "text_feature_count": len(text_features),
            "recovered_lists_match": recovered_lists_match,
        }
    )


parser_validation = pd.DataFrame(parser_validation_rows)

parser_validation_summary = pd.Series(
    {
        "records_checked": len(parser_validation),
        "expected_features_per_record": (
            expected_primary_feature_count
        ),
        "structured_records_with_expected_features": int(
            parser_validation["structured_feature_count"]
            .eq(expected_primary_feature_count)
            .sum()
        ),
        "text_records_with_expected_features": int(
            parser_validation["text_feature_count"]
            .eq(expected_primary_feature_count)
            .sum()
        ),
        "recovered_representation_pairs_matching": int(
            parser_validation["recovered_lists_match"].sum()
        ),
        "grounding_reference_records": len(
            grounding_reference_by_sample
        ),
    },
    name="value",
)

parser_validation_summary

records_checked                              200
expected_features_per_record                  46
structured_records_with_expected_features    200
text_records_with_expected_features          200
recovered_representation_pairs_matching      200
grounding_reference_records                  200
Name: value, dtype: int64

### Enforce the parser-validation requirements

The recovered feature–value mappings will be used as the reference for assessing LLM citations. Consequently, partial parser success is not sufficient: every expected record must pass.

The assertions below convert the parser requirements into an execution gate. If any requirement fails, response construction and inference must not proceed.

In [12]:
assert len(parser_validation) == expected_record_count, (
    "Parser validation did not cover every expected record."
)

assert (
    parser_validation["structured_feature_count"]
    .eq(expected_primary_feature_count)
    .all()
), (
    "At least one structured input did not recover the expected "
    "number of features."
)

assert (
    parser_validation["text_feature_count"]
    .eq(expected_primary_feature_count)
    .all()
), (
    "At least one deterministic-text input did not recover the expected "
    "number of features."
)

assert parser_validation["recovered_lists_match"].all(), (
    "At least one paired record produced different recovered feature lists."
)

assert len(grounding_reference_by_sample) == expected_record_count, (
    "The grounding reference does not contain every expected sample."
)

assert set(grounding_reference_by_sample) == structured_sample_id_set, (
    "The grounding-reference sample IDs do not match the validated input IDs."
)

assert all(
    len(feature_lookup) == expected_primary_feature_count
    for feature_lookup in grounding_reference_by_sample.values()
), (
    "At least one grounding-reference record does not contain the expected "
    "number of distinct features."
)

print(
    "Parser gate passed: all "
    f"{expected_record_count} paired records recover the same "
    f"{expected_primary_feature_count} ordered feature–value pairs."
)

Parser gate passed: all 200 paired records recover the same 46 ordered feature–value pairs.


## Validate LLM responses

A response can be syntactically valid while still being unsupported by the supplied record. Response assessment is therefore divided into three levels.

### 1. JSON validity

The raw response must decode to one JSON object.

### 2. Schema validity

The decoded object must:

- contain exactly `predicted_label` and `cited_features`;
- use either `Benign` or `DoS`;
- contain exactly five citation objects;
- give every citation exactly `feature_name`, `observed_value` and `rationale`;
- use strings for all citation fields; and
- provide a non-empty rationale.

### 3. Grounding validity

The five citations must:

- use five distinct feature names;
- cite only features supplied in the target record; and
- copy the corresponding observed values exactly.

Schema and grounding results are retained separately. This distinction allows later analysis to report formatting failures, invalid feature citations and value-copying errors independently.

The validator assesses explicit response content. It cannot establish that a plausible rationale faithfully represents the model's hidden internal computation, nor can it fully judge the semantic quality of each rationale automatically.

In [13]:
ALLOWED_PREDICTED_LABELS = {"Benign", "DoS"}
EXPECTED_TOP_LEVEL_RESPONSE_FIELDS = {
    "predicted_label",
    "cited_features",
}
EXPECTED_CITATION_FIELDS = {
    "feature_name",
    "observed_value",
    "rationale",
}
EXPECTED_CITATION_COUNT = 5


def validate_llm_response(
    raw_response: str,
    *,
    sample_id: str,
) -> dict:
    """
    Validate one raw LLM response against the locked protocol and target input.

    Validation is divided into JSON validity, schema validity and grounding
    validity. The function records all detectable errors instead of stopping
    after the first one, which supports later error-rate analysis.

    Parameters
    ----------
    raw_response:
        The unmodified text returned by the LLM provider.
    sample_id:
        The outer experiment identifier used to select the corresponding
        feature–value grounding reference. It is not sent to the LLM.

    Returns
    -------
    dict
        Validation flags, diagnostic counts, error codes, human-readable
        messages and the parsed response when JSON decoding succeeds.

    Raises
    ------
    TypeError
        If `raw_response` or `sample_id` is not a string.
    KeyError
        If `sample_id` is absent from the validated grounding reference.
    """
    if not isinstance(raw_response, str):
        raise TypeError(
            f"raw_response must be a string, not "
            f"{type(raw_response).__name__}."
        )

    if not isinstance(sample_id, str) or not sample_id:
        raise TypeError("sample_id must be a non-empty string.")

    if sample_id not in grounding_reference_by_sample:
        raise KeyError(
            f"No grounding reference exists for sample {sample_id!r}."
        )

    error_codes: list[str] = []
    error_messages: list[str] = []

    def record_error(code: str, message: str) -> None:
        """Add one error while preventing duplicate error codes."""
        if code not in error_codes:
            error_codes.append(code)
            error_messages.append(message)

    result = {
        "sample_id": sample_id,
        "json_valid": False,
        "schema_valid": False,
        "grounding_valid": False,
        "overall_valid": False,
        "citation_count": None,
        "distinct_cited_feature_count": None,
        "valid_feature_name_count": 0,
        "matching_observed_value_count": 0,
        "error_codes": error_codes,
        "error_messages": error_messages,
        "parsed_response": None,
    }

    try:
        parsed_response = json.loads(raw_response)
    except json.JSONDecodeError as exc:
        record_error(
            "invalid_json",
            f"The response is not valid JSON: {exc.msg}.",
        )
        return result

    result["json_valid"] = True
    result["parsed_response"] = parsed_response

    if not isinstance(parsed_response, dict):
        record_error(
            "top_level_not_object",
            "The decoded response must be a JSON object.",
        )
        return result

    schema_valid = True

    top_level_fields = set(parsed_response)

    if top_level_fields != EXPECTED_TOP_LEVEL_RESPONSE_FIELDS:
        schema_valid = False

        missing_fields = sorted(
            EXPECTED_TOP_LEVEL_RESPONSE_FIELDS - top_level_fields
        )
        extra_fields = sorted(
            top_level_fields - EXPECTED_TOP_LEVEL_RESPONSE_FIELDS
        )

        if missing_fields:
            record_error(
                "missing_top_level_fields",
                f"Missing top-level fields: {missing_fields}.",
            )

        if extra_fields:
            record_error(
                "extra_top_level_fields",
                f"Unexpected top-level fields: {extra_fields}.",
            )

    predicted_label = parsed_response.get("predicted_label")

    if predicted_label not in ALLOWED_PREDICTED_LABELS:
        schema_valid = False
        record_error(
            "invalid_predicted_label",
            "predicted_label must be exactly 'Benign' or 'DoS'.",
        )

    cited_features = parsed_response.get("cited_features")

    if not isinstance(cited_features, list):
        schema_valid = False
        record_error(
            "cited_features_not_array",
            "cited_features must be a JSON array.",
        )
        result["schema_valid"] = schema_valid
        return result

    result["citation_count"] = len(cited_features)

    if len(cited_features) != EXPECTED_CITATION_COUNT:
        schema_valid = False
        record_error(
            "incorrect_citation_count",
            "cited_features must contain exactly "
            f"{EXPECTED_CITATION_COUNT} items.",
        )

    structurally_valid_citations = []

    for rank, citation in enumerate(cited_features, start=1):
        citation_is_structurally_valid = True

        if not isinstance(citation, dict):
            schema_valid = False
            record_error(
                "citation_not_object",
                f"Citation at rank {rank} is not a JSON object.",
            )
            continue

        citation_fields = set(citation)

        if citation_fields != EXPECTED_CITATION_FIELDS:
            schema_valid = False
            citation_is_structurally_valid = False
            record_error(
                "invalid_citation_fields",
                "Each citation must contain exactly feature_name, "
                "observed_value and rationale.",
            )

        feature_name = citation.get("feature_name")
        observed_value = citation.get("observed_value")
        rationale = citation.get("rationale")

        if not isinstance(feature_name, str) or not feature_name:
            schema_valid = False
            citation_is_structurally_valid = False
            record_error(
                "invalid_feature_name_type",
                "Every feature_name must be a non-empty string.",
            )

        if not isinstance(observed_value, str) or not observed_value:
            schema_valid = False
            citation_is_structurally_valid = False
            record_error(
                "invalid_observed_value_type",
                "Every observed_value must be a non-empty string.",
            )

        if not isinstance(rationale, str) or not rationale.strip():
            schema_valid = False
            citation_is_structurally_valid = False
            record_error(
                "empty_or_invalid_rationale",
                "Every rationale must be a non-empty string.",
            )

        if citation_is_structurally_valid:
            structurally_valid_citations.append(citation)

    result["schema_valid"] = schema_valid

    # Grounding is assessed for every structurally recoverable citation, even
    # when another part of the response has a schema error. This preserves
    # useful diagnostics without treating the whole response as valid.
    cited_feature_names = [
        citation["feature_name"]
        for citation in structurally_valid_citations
    ]

    result["distinct_cited_feature_count"] = len(
        set(cited_feature_names)
    )

    grounding_valid = True

    if (
        len(cited_feature_names) != EXPECTED_CITATION_COUNT
        or len(set(cited_feature_names)) != EXPECTED_CITATION_COUNT
    ):
        grounding_valid = False
        record_error(
            "duplicate_or_incomplete_feature_ranking",
            "The response must cite five distinct feature names.",
        )

    grounding_reference = grounding_reference_by_sample[sample_id]

    for citation in structurally_valid_citations:
        feature_name = citation["feature_name"]
        observed_value = citation["observed_value"]

        if feature_name not in grounding_reference:
            grounding_valid = False
            record_error(
                "unsupported_feature_name",
                "At least one cited feature is absent from the "
                "supplied record.",
            )
            continue

        result["valid_feature_name_count"] += 1

        expected_value = grounding_reference[feature_name]

        if observed_value != expected_value:
            grounding_valid = False
            record_error(
                "mismatched_observed_value",
                "At least one cited observed value does not match the "
                "supplied record.",
            )
            continue

        result["matching_observed_value_count"] += 1

    if (
        result["valid_feature_name_count"] != EXPECTED_CITATION_COUNT
        or result["matching_observed_value_count"]
        != EXPECTED_CITATION_COUNT
    ):
        grounding_valid = False

    result["grounding_valid"] = grounding_valid
    result["overall_valid"] = schema_valid and grounding_valid

    return result

### Construct one valid synthetic response

Before processing any real LLM output, the validator is tested using a synthetic response assembled from the first five actual features of `pilot_001`.

The synthetic rationales are placeholders used only to test response mechanics. They are not model predictions and must not be interpreted as anomaly-analysis findings.

In [14]:
validation_test_sample_id = structured_records[0]["sample_id"]

valid_synthetic_response_object = {
    "predicted_label": "Benign",
    "cited_features": [
        {
            "feature_name": feature["feature_name"],
            "observed_value": feature["observed_value"],
            "rationale": (
                "Synthetic validator test using a supplied feature."
            ),
        }
        for feature in first_structured_features[:5]
    ],
}

valid_synthetic_response_text = json.dumps(
    valid_synthetic_response_object,
    ensure_ascii=False,
    separators=(",", ":"),
)

valid_synthetic_validation = validate_llm_response(
    valid_synthetic_response_text,
    sample_id=validation_test_sample_id,
)

pd.Series(
    {
        key: value
        for key, value in valid_synthetic_validation.items()
        if key not in {
            "parsed_response",
            "error_messages",
        }
    },
    name="value",
)

sample_id                        pilot_001
json_valid                            True
schema_valid                          True
grounding_valid                       True
overall_valid                         True
citation_count                           5
distinct_cited_feature_count             5
valid_feature_name_count                 5
matching_observed_value_count            5
error_codes                             []
Name: value, dtype: object

### Test the validator with deliberately invalid responses

A validator must demonstrate both acceptance of valid output and rejection of known-invalid output.

Seven negative test cases are derived from the valid synthetic response. Each case introduces one targeted defect:

1. invalid JSON syntax;
2. an unsupported prediction label;
3. fewer than five citations;
4. a feature name absent from the supplied record;
5. an observed value that does not match the supplied record;
6. a duplicated feature citation;
7. an empty rationale; and
8. an unexpected top-level field.

Deep copies are used so that modifying one test case cannot alter the valid baseline or another test case.

These are mechanical protocol tests. They do not evaluate whether a feature rationale is scientifically persuasive.

In [15]:
def serialise_test_response(response_object: dict) -> str:
    """
    Serialise a synthetic response using the same compact JSON convention
    expected from the LLM.

    Parameters
    ----------
    response_object:
        A JSON-compatible synthetic response dictionary.

    Returns
    -------
    str
        Compact JSON text suitable for the response validator.
    """
    return json.dumps(
        response_object,
        ensure_ascii=False,
        separators=(",", ":"),
    )


invalid_label_object = copy.deepcopy(
    valid_synthetic_response_object
)
invalid_label_object["predicted_label"] = "Attack"


too_few_citations_object = copy.deepcopy(
    valid_synthetic_response_object
)
too_few_citations_object["cited_features"] = (
    too_few_citations_object["cited_features"][:4]
)


unsupported_feature_object = copy.deepcopy(
    valid_synthetic_response_object
)
unsupported_feature_object["cited_features"][0][
    "feature_name"
] = "NOT_A_SUPPLIED_FEATURE"
unsupported_feature_object["cited_features"][0][
    "observed_value"
] = "0"


mismatched_value_object = copy.deepcopy(
    valid_synthetic_response_object
)
mismatched_value_object["cited_features"][0][
    "observed_value"
] = "INTENTIONALLY_WRONG_VALUE"


duplicate_feature_object = copy.deepcopy(
    valid_synthetic_response_object
)
duplicate_feature_object["cited_features"][-1] = copy.deepcopy(
    duplicate_feature_object["cited_features"][0]
)


empty_rationale_object = copy.deepcopy(
    valid_synthetic_response_object
)
empty_rationale_object["cited_features"][0]["rationale"] = "   "


extra_field_object = copy.deepcopy(
    valid_synthetic_response_object
)
extra_field_object["confidence"] = 0.95


response_validator_test_cases = [
    {
        "case": "valid_response",
        "raw_response": valid_synthetic_response_text,
        "expected_error_code": None,
        "expected_overall_valid": True,
    },
    {
        "case": "invalid_json",
        "raw_response": '{"predicted_label":"Benign"',
        "expected_error_code": "invalid_json",
        "expected_overall_valid": False,
    },
    {
        "case": "invalid_label",
        "raw_response": serialise_test_response(
            invalid_label_object
        ),
        "expected_error_code": "invalid_predicted_label",
        "expected_overall_valid": False,
    },
    {
        "case": "too_few_citations",
        "raw_response": serialise_test_response(
            too_few_citations_object
        ),
        "expected_error_code": "incorrect_citation_count",
        "expected_overall_valid": False,
    },
    {
        "case": "unsupported_feature",
        "raw_response": serialise_test_response(
            unsupported_feature_object
        ),
        "expected_error_code": "unsupported_feature_name",
        "expected_overall_valid": False,
    },
    {
        "case": "mismatched_value",
        "raw_response": serialise_test_response(
            mismatched_value_object
        ),
        "expected_error_code": "mismatched_observed_value",
        "expected_overall_valid": False,
    },
    {
        "case": "duplicate_feature",
        "raw_response": serialise_test_response(
            duplicate_feature_object
        ),
        "expected_error_code": (
            "duplicate_or_incomplete_feature_ranking"
        ),
        "expected_overall_valid": False,
    },
    {
        "case": "empty_rationale",
        "raw_response": serialise_test_response(
            empty_rationale_object
        ),
        "expected_error_code": "empty_or_invalid_rationale",
        "expected_overall_valid": False,
    },
    {
        "case": "extra_top_level_field",
        "raw_response": serialise_test_response(
            extra_field_object
        ),
        "expected_error_code": "extra_top_level_fields",
        "expected_overall_valid": False,
    },
]

### Execute the response-validator tests

Each synthetic response is passed through the same validator that will later assess real LLM outputs.

A test passes only when:

- the expected error code is detected, or the valid case has no errors; and
- the resulting overall-valid flag matches the expected result.

The test suite stops the notebook if any expected behaviour is not observed.

In [16]:
response_validator_test_rows = []

for test_case in response_validator_test_cases:
    validation_result = validate_llm_response(
        test_case["raw_response"],
        sample_id=validation_test_sample_id,
    )

    expected_error_code = test_case["expected_error_code"]

    if expected_error_code is None:
        expected_error_detected = (
            validation_result["error_codes"] == []
        )
    else:
        expected_error_detected = (
            expected_error_code
            in validation_result["error_codes"]
        )

    overall_valid_matches_expectation = (
        validation_result["overall_valid"]
        == test_case["expected_overall_valid"]
    )

    test_passed = (
        expected_error_detected
        and overall_valid_matches_expectation
    )

    response_validator_test_rows.append(
        {
            "case": test_case["case"],
            "json_valid": validation_result["json_valid"],
            "schema_valid": validation_result["schema_valid"],
            "grounding_valid": validation_result[
                "grounding_valid"
            ],
            "overall_valid": validation_result["overall_valid"],
            "expected_error_code": expected_error_code,
            "detected_error_codes": ", ".join(
                validation_result["error_codes"]
            ),
            "test_passed": test_passed,
        }
    )


response_validator_test_results = pd.DataFrame(
    response_validator_test_rows
).set_index("case")

display(response_validator_test_results)

assert response_validator_test_results["test_passed"].all(), (
    "At least one response-validator test did not behave as expected."
)

print(
    "Response-validator gate passed: all "
    f"{len(response_validator_test_results)} synthetic test cases "
    "behaved as expected."
)

,json_valid,schema_valid,grounding_valid,overall_valid,expected_error_code,detected_error_codes,test_passed
case,,,,,,,
valid_response,True,True,True,True,NaN,,True
invalid_json,False,False,False,False,invalid_json,invalid_json,True
invalid_label,True,False,True,False,invalid_predicted_label,invalid_predicted_label,True
too_few_citations,True,False,False,False,incorrect_citation_count,"incorrect_citation_count, duplicate_or_incompl...",True
unsupported_feature,True,True,False,False,unsupported_feature_name,unsupported_feature_name,True
mismatched_value,True,True,False,False,mismatched_observed_value,mismatched_observed_value,True
duplicate_feature,True,True,False,False,duplicate_or_incomplete_feature_ranking,duplicate_or_incomplete_feature_ranking,True
empty_rationale,True,False,False,False,empty_or_invalid_rationale,"empty_or_invalid_rationale, duplicate_or_incom...",True
extra_top_level_field,True,False,True,False,extra_top_level_fields,extra_top_level_fields,True


Response-validator gate passed: all 9 synthetic test cases behaved as expected.


### Interpretation of the response-validator tests

All nine synthetic test cases behaved as expected.

The validator accepted the valid baseline and rejected each deliberately invalid response. It also distinguished structural failures from grounding failures. For example:

- an unsupported feature name can satisfy the JSON schema while failing grounding validation;
- an incorrect citation count fails both the fixed output structure and the complete five-feature ranking requirement; and
- an unexpected confidence field fails schema validation even if all cited features remain grounded.

These tests demonstrate the validator's expected behaviour for the predefined mechanical error cases. They do not prove that every possible malformed response will be detected, nor do they assess whether a natural-language rationale is scientifically persuasive or causally faithful.

## Construct the provider-independent request contract

Before selecting an API provider or model, the experiment defines a provider-independent request contract.

Each request contains exactly three experimental components:

1. the common system instruction;
2. one representation-specific user message; and
3. the common output schema.

Provider-specific settings such as model identifier, temperature support, seed support, token-limit parameter names and API response format are deliberately excluded at this stage. They will be locked only after the selected provider and model capabilities have been verified.

Outer experiment metadata such as `sample_id`, feature-set identifier and payload hash is retained in the local request registry but is not included in the content sent to the LLM.

### Implementation map

This section prepares the experimental content that will later be translated into a provider-specific API request.

The code follows four steps:

1. `stable_json_sha256()` converts protocol components into stable content hashes so that later changes can be detected.
2. `build_logical_request_contract()` combines the common instruction, one record representation and the common response schema.
3. The first structured/text pair is inspected in detail to make the comparison understandable.
4. The same requirements are applied to all 200 pairs and enforced as a blocking gate.

A logical request contract is not yet an API request. It contains no model name, API key, provider endpoint, temperature or token limit. Those settings will be added only after the provider and model are selected.

In [17]:
def stable_json_sha256(value: object) -> str:
    """
    Calculate a reproducible SHA-256 hash for a JSON-compatible object.

    Dictionary keys are sorted and unnecessary whitespace is removed so that
    the hash reflects content rather than Python dictionary display format.

    Parameters
    ----------
    value:
        A JSON-compatible object.

    Returns
    -------
    str
        The lowercase hexadecimal SHA-256 digest.
    """
    canonical_json = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        canonical_json.encode("utf-8")
    ).hexdigest()


def build_logical_request_contract(
    model_input: str,
) -> dict:
    """
    Build the provider-independent content contract for one LLM request.

    The function does not add sample identifiers, labels, dataset provenance
    or provider-specific inference settings.

    Parameters
    ----------
    model_input:
        One validated structured or deterministic-text representation.

    Returns
    -------
    dict
        The common system instruction, representation-specific user message
        and locked output schema.
    """
    return {
        "system_instruction": COMMON_SYSTEM_INSTRUCTION,
        "user_message": build_user_message(model_input),
        "output_schema": copy.deepcopy(LLM_OUTPUT_SCHEMA),
    }


first_structured_request_contract = build_logical_request_contract(
    structured_records[0]["model_input"]
)
first_text_request_contract = build_logical_request_contract(
    text_records[0]["model_input"]
)

### Compare the first paired request contracts

The two contracts must use identical instructions and output schemas. Their user messages must differ because the enclosed record representation differs.

The comparison also verifies that:

- the paired canonical-payload hashes match;
- neither request contains the outer `sample_id`;
- the 46 permitted feature names are identical; and
- neither recovered feature list contains the ground-truth fields `Label` or `Attack`.

Character counts are reported descriptively. They are not expected to match because representation length and tokenisation are part of the representation treatment.

In [18]:
first_sample_id = structured_records[0]["sample_id"]

first_structured_feature_names = {
    feature["feature_name"]
    for feature in first_structured_features
}
first_text_feature_names = {
    feature["feature_name"]
    for feature in first_text_features
}

ground_truth_field_names = {"Label", "Attack"}

first_request_contract_comparison = pd.Series(
    {
        "sample_id": first_sample_id,
        "system_instructions_match": (
            first_structured_request_contract["system_instruction"]
            == first_text_request_contract["system_instruction"]
        ),
        "output_schemas_match": (
            first_structured_request_contract["output_schema"]
            == first_text_request_contract["output_schema"]
        ),
        "user_messages_differ": (
            first_structured_request_contract["user_message"]
            != first_text_request_contract["user_message"]
        ),
        "canonical_payload_hashes_match": (
            structured_records[0]["canonical_payload_sha256"]
            == text_records[0]["canonical_payload_sha256"]
        ),
        "structured_feature_names_match_text": (
            first_structured_feature_names
            == first_text_feature_names
        ),
        "structured_ground_truth_fields_present": bool(
            first_structured_feature_names
            & ground_truth_field_names
        ),
        "text_ground_truth_fields_present": bool(
            first_text_feature_names
            & ground_truth_field_names
        ),
        "sample_id_present_in_structured_request": (
            first_sample_id
            in first_structured_request_contract["user_message"]
        ),
        "sample_id_present_in_text_request": (
            first_sample_id
            in first_text_request_contract["user_message"]
        ),
        "structured_user_message_characters": len(
            first_structured_request_contract["user_message"]
        ),
        "text_user_message_characters": len(
            first_text_request_contract["user_message"]
        ),
        "system_instruction_sha256": stable_json_sha256(
            first_structured_request_contract["system_instruction"]
        ),
        "output_schema_sha256": stable_json_sha256(
            first_structured_request_contract["output_schema"]
        ),
    },
    name="value",
)

first_request_contract_comparison

sample_id                                                                          pilot_001
system_instructions_match                                                               True
output_schemas_match                                                                    True
user_messages_differ                                                                    True
canonical_payload_hashes_match                                                          True
structured_feature_names_match_text                                                     True
structured_ground_truth_fields_present                                                 False
text_ground_truth_fields_present                                                       False
sample_id_present_in_structured_request                                                False
sample_id_present_in_text_request                                                      False
structured_user_message_characters                                    

### Interpretation of the first request-contract comparison

For `pilot_001`, the structured and deterministic-text contracts use identical system instructions and output schemas. Their user messages differ, as expected, because one contains JSON and the other contains deterministic text.

The matching canonical-payload hashes and recovered feature names show that this difference is limited to presentation rather than underlying feature information.

Neither contract contains the outer `sample_id`, and neither recovered feature list contains `Label` or `Attack`. The different character counts are descriptive properties of the two representations and are not treated as validation failures.

This result verifies the first pair only. The following validation extends the same checks to all 200 pairs.

### Validate all paired request contracts

The first-pair comparison is extended to all 200 samples.

For every pair, the common instructions and output schema must remain identical. The representation-specific user messages must remain different while being linked to the same canonical payload.

This is the final provider-independent fairness gate before model and inference settings are selected.

In [19]:
request_contract_validation_rows = []

for structured_record, text_record in zip(
    structured_records,
    text_records,
    strict=True,
):
    sample_id = structured_record["sample_id"]

    structured_contract = build_logical_request_contract(
        structured_record["model_input"]
    )
    text_contract = build_logical_request_contract(
        text_record["model_input"]
    )

    structured_feature_names = set(
        grounding_reference_by_sample[sample_id]
    )

    request_contract_validation_rows.append(
        {
            "sample_id": sample_id,
            "system_instructions_match": (
                structured_contract["system_instruction"]
                == text_contract["system_instruction"]
            ),
            "output_schemas_match": (
                structured_contract["output_schema"]
                == text_contract["output_schema"]
            ),
            "user_messages_differ": (
                structured_contract["user_message"]
                != text_contract["user_message"]
            ),
            "canonical_payload_hashes_match": (
                structured_record["canonical_payload_sha256"]
                == text_record["canonical_payload_sha256"]
            ),
            "ground_truth_fields_absent": not bool(
                structured_feature_names & ground_truth_field_names
            ),
            "sample_id_absent_from_structured_request": (
                sample_id not in structured_contract["user_message"]
            ),
            "sample_id_absent_from_text_request": (
                sample_id not in text_contract["user_message"]
            ),
        }
    )


request_contract_validation = pd.DataFrame(
    request_contract_validation_rows
)

request_contract_validation_summary = pd.Series(
    {
        "request_pairs_checked": len(
            request_contract_validation
        ),
        "matching_system_instructions": int(
            request_contract_validation[
                "system_instructions_match"
            ].sum()
        ),
        "matching_output_schemas": int(
            request_contract_validation[
                "output_schemas_match"
            ].sum()
        ),
        "differing_user_messages": int(
            request_contract_validation[
                "user_messages_differ"
            ].sum()
        ),
        "matching_canonical_payload_hashes": int(
            request_contract_validation[
                "canonical_payload_hashes_match"
            ].sum()
        ),
        "pairs_without_ground_truth_fields": int(
            request_contract_validation[
                "ground_truth_fields_absent"
            ].sum()
        ),
        "structured_requests_without_sample_id": int(
            request_contract_validation[
                "sample_id_absent_from_structured_request"
            ].sum()
        ),
        "text_requests_without_sample_id": int(
            request_contract_validation[
                "sample_id_absent_from_text_request"
            ].sum()
        ),
    },
    name="value",
)

request_contract_validation_summary

request_pairs_checked                    200
matching_system_instructions             200
matching_output_schemas                  200
differing_user_messages                  200
matching_canonical_payload_hashes        200
pairs_without_ground_truth_fields        200
structured_requests_without_sample_id    200
text_requests_without_sample_id          200
Name: value, dtype: int64

### Interpretation of the full request-contract validation

All 200 paired contracts passed the predefined comparison checks:

- all 200 use the same system instruction;
- all 200 use the same output schema;
- all 200 contain different rendered user messages;
- all 200 are linked to matching canonical payloads;
- all 200 exclude the ground-truth fields; and
- all 400 individual requests exclude the outer sample identifier.

Therefore, within the scope of the validated request content, the planned experimental difference is limited to the structured versus deterministic-text representation.

This check does not yet establish equality of provider-side processing, token counts, latency or stochastic model behaviour. Those factors require provider-specific configuration and run metadata.

### Enforce the provider-independent request-contract requirements

All paired request contracts must pass before provider-specific API code or inference settings are introduced.

In [20]:
request_contract_requirements = [
    "system_instructions_match",
    "output_schemas_match",
    "user_messages_differ",
    "canonical_payload_hashes_match",
    "ground_truth_fields_absent",
    "sample_id_absent_from_structured_request",
    "sample_id_absent_from_text_request",
]

assert len(request_contract_validation) == expected_record_count, (
    "Request-contract validation did not cover every expected pair."
)

assert request_contract_validation[
    request_contract_requirements
].all().all(), (
    "At least one paired request contract violates the locked requirements."
)

print(
    "Request-contract gate passed: all "
    f"{expected_record_count} pairs use identical instructions and schemas, "
    "contain no sample IDs or ground-truth fields, and differ only in the "
    "validated record representation."
)

Request-contract gate passed: all 200 pairs use identical instructions and schemas, contain no sample IDs or ground-truth fields, and differ only in the validated record representation.


## Persist the provider-independent protocol

The validated task instruction and response schema are now saved as tracked configuration files.

Separating protocol content from provider-specific settings has two benefits:

1. the scientific task definition can be reviewed and version-controlled before any API is selected; and
2. changing an API adapter cannot silently change the classification task or required output.

This cell writes two files:

- `llm_output_schema.json`, containing the strict response schema; and
- `llm_protocol_primary_46.json`, documenting the paired conditions, common instruction, response requirements and leakage controls.

No API credentials, model outputs or private ground-truth labels are written to these files.

In [21]:
LLM_OUTPUT_SCHEMA_PATH = (
    PROJECT_ROOT / "configs" / "llm_output_schema.json"
)
LLM_PROTOCOL_PATH = (
    PROJECT_ROOT / "configs" / "llm_protocol_primary_46.json"
)


def calculate_file_sha256(path: Path) -> str:
    """
    Calculate the SHA-256 digest of a file without modifying it.

    The file is read in fixed-size binary chunks so the function can also be
    used later for larger artifacts without loading an entire file into memory.

    Parameters
    ----------
    path:
        Existing file whose content should be hashed.

    Returns
    -------
    str
        Lowercase hexadecimal SHA-256 digest.

    Raises
    ------
    FileNotFoundError
        If the requested file does not exist.
    """
    if not path.is_file():
        raise FileNotFoundError(
            f"Cannot hash missing file: {path}"
        )

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(65_536), b""):
            digest.update(chunk)

    return digest.hexdigest()


# Store the schema exactly as validated in memory. deepcopy prevents the saved
# protocol object from sharing a mutable nested schema with notebook variables.
output_schema_to_save = copy.deepcopy(LLM_OUTPUT_SCHEMA)

output_schema_sha256 = stable_json_sha256(
    output_schema_to_save
)

llm_protocol_primary_46 = {
    "protocol_version": "0.1.0",
    "protocol_status": "provider_independent_locked",
    "task": {
        "name": "binary_network_flow_anomaly_detection",
        "allowed_labels": ["Benign", "DoS"],
        "decision_mode": "forced_binary_classification",
    },
    "feature_set": {
        "feature_set_id": "primary_46",
        "feature_count": expected_primary_feature_count,
        "feature_config_path": (
            "configs/feature_set_primary_46.json"
        ),
        "feature_config_sha256": calculate_file_sha256(
            FEATURE_SET_PATH
        ),
    },
    "evaluation_inputs": {
        "record_count": expected_record_count,
        "conditions": [
            "structured",
            "deterministic_text",
        ],
        "paired_records": True,
        "same_canonical_payload_required": True,
    },
    "instructions": {
        "system_instruction": COMMON_SYSTEM_INSTRUCTION,
        "record_opening_delimiter": RECORD_OPENING_DELIMITER,
        "record_closing_delimiter": RECORD_CLOSING_DELIMITER,
    },
    "response_policy": {
        "schema_path": "configs/llm_output_schema.json",
        "schema_content_sha256": output_schema_sha256,
        "required_citation_count": EXPECTED_CITATION_COUNT,
        "distinct_feature_names_required": True,
        "exact_feature_name_copy_required": True,
        "exact_observed_value_copy_required": True,
        "self_reported_confidence_included": False,
    },
    "information_controls": {
        "ground_truth_loaded_during_inference": False,
        "label_field_in_model_input": False,
        "attack_field_in_model_input": False,
        "sample_id_in_model_input": False,
        "dataset_identity_provided": False,
        "class_balance_provided": False,
        "thresholds_or_normal_ranges_provided": False,
        "worked_examples_provided": False,
    },
    "provider_specific_settings": {
        "status": "not_yet_locked",
        "note": (
            "Provider, model identifier and supported inference parameters "
            "must be verified and recorded before any API call."
        ),
    },
}


# Write human-readable, deterministic JSON. Sorting keys makes diffs and file
# hashes stable across repeated executions with unchanged protocol content.
LLM_OUTPUT_SCHEMA_PATH.write_text(
    json.dumps(
        output_schema_to_save,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

LLM_PROTOCOL_PATH.write_text(
    json.dumps(
        llm_protocol_primary_46,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

saved_protocol_files = pd.DataFrame(
    [
        {
            "artifact": LLM_OUTPUT_SCHEMA_PATH.name,
            "relative_path": LLM_OUTPUT_SCHEMA_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
            "size_bytes": LLM_OUTPUT_SCHEMA_PATH.stat().st_size,
            "sha256": calculate_file_sha256(
                LLM_OUTPUT_SCHEMA_PATH
            ),
        },
        {
            "artifact": LLM_PROTOCOL_PATH.name,
            "relative_path": LLM_PROTOCOL_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
            "size_bytes": LLM_PROTOCOL_PATH.stat().st_size,
            "sha256": calculate_file_sha256(
                LLM_PROTOCOL_PATH
            ),
        },
    ]
).set_index("artifact")

saved_protocol_files

,relative_path,size_bytes,sha256
artifact,,,
llm_output_schema.json,configs/llm_output_schema.json,1482,1103a917ce1cad918ab4a94c5d94d8a0299400749ca9e0...
llm_protocol_primary_46.json,configs/llm_protocol_primary_46.json,2816,3ff01d1b7fa91f0b38d03a4324034e26e71c3288b74cd3...


### Reload and verify the saved protocol

The two configuration files are reloaded from disk and compared with the validated in-memory objects.

This detects incomplete writes, accidental encoding changes and mismatches between the notebook protocol and the tracked files.

In [22]:
with LLM_OUTPUT_SCHEMA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    reloaded_output_schema = json.load(handle)

with LLM_PROTOCOL_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    reloaded_llm_protocol = json.load(handle)


saved_protocol_verification = pd.Series(
    {
        "output_schema_reloads_exactly": (
            reloaded_output_schema == output_schema_to_save
        ),
        "llm_protocol_reloads_exactly": (
            reloaded_llm_protocol == llm_protocol_primary_46
        ),
        "saved_schema_hash_matches_protocol": (
            stable_json_sha256(reloaded_output_schema)
            == reloaded_llm_protocol[
                "response_policy"
            ]["schema_content_sha256"]
        ),
        "saved_feature_count": reloaded_llm_protocol[
            "feature_set"
        ]["feature_count"],
        "saved_record_count": reloaded_llm_protocol[
            "evaluation_inputs"
        ]["record_count"],
        "provider_settings_locked": (
            reloaded_llm_protocol[
                "provider_specific_settings"
            ]["status"]
            == "locked"
        ),
    },
    name="value",
)

saved_protocol_verification

output_schema_reloads_exactly          True
llm_protocol_reloads_exactly           True
saved_schema_hash_matches_protocol     True
saved_feature_count                      46
saved_record_count                      200
provider_settings_locked              False
Name: value, dtype: object

### Enforce the saved-protocol requirements

The provider-independent artifacts must reload exactly and contain the expected feature and record counts. Provider settings are intentionally required to remain unlocked at this stage.

In [23]:
assert reloaded_output_schema == output_schema_to_save, (
    "The saved output schema differs from the validated in-memory schema."
)

assert reloaded_llm_protocol == llm_protocol_primary_46, (
    "The saved LLM protocol differs from the in-memory protocol."
)

assert (
    stable_json_sha256(reloaded_output_schema)
    == reloaded_llm_protocol[
        "response_policy"
    ]["schema_content_sha256"]
), "The saved output-schema hash does not match the protocol reference."

assert reloaded_llm_protocol[
    "feature_set"
]["feature_count"] == expected_primary_feature_count, (
    "The saved protocol contains an incorrect feature count."
)

assert reloaded_llm_protocol[
    "evaluation_inputs"
]["record_count"] == expected_record_count, (
    "The saved protocol contains an incorrect record count."
)

assert reloaded_llm_protocol[
    "provider_specific_settings"
]["status"] == "not_yet_locked", (
    "Provider settings were unexpectedly locked before model selection."
)

print(
    "Saved-protocol gate passed: the provider-independent schema and "
    "primary-46 LLM protocol reload exactly and remain ready for "
    "provider-specific configuration."
)

Saved-protocol gate passed: the provider-independent schema and primary-46 LLM protocol reload exactly and remain ready for provider-specific configuration.


### Verify the local SDK and gateway credential

Before any external request is made, this pre-flight check verifies that the
current Python environment contains the required client library and can access
the UoA Gateway credential through the `UOA_API_KEY` environment variable.

The credential value is deliberately neither displayed nor written to disk.
Only its presence is reported. This cell performs no network request and
therefore consumes no gateway quota.

In [24]:
import os
import sys
from importlib import metadata


def get_installed_package_version(package_name):
    """
    Return the installed version of a Python package.

    Parameters
    ----------
    package_name : str
        Distribution name recorded by the Python package manager.

    Returns
    -------
    str or None
        Installed version, or None when the package is unavailable.

    Notes
    -----
    This function inspects local package metadata only. It does not import
    the target package and does not make any network request.
    """
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return None


# Read only whether the credential exists. Never display its actual value.
uoa_api_key_present = bool(os.environ.get("UOA_API_KEY", "").strip())

# Record the local environment needed to reproduce the later API calls.
gateway_environment_check = pd.Series(
    {
        "python_executable": sys.executable,
        "python_version": sys.version.split()[0],
        "openai_sdk_version": get_installed_package_version("openai"),
        "uoa_api_key_present": uoa_api_key_present,
        "network_request_made": False,
    },
    name="value",
)

display(gateway_environment_check)

python_executable       /opt/anaconda3/envs/compsci742-a2/bin/python
python_version                                               3.11.14
openai_sdk_version                                             3.6.0
uoa_api_key_present                                             True
network_request_made                                           False
Name: value, dtype: object

### Install the OpenAI-compatible Python client

The UoA Agentic Gateway exposes an OpenAI-compatible API. The `openai` Python
package is therefore used only as the client library that constructs and sends
requests to the University gateway. Installing this package does not select an
OpenAI-hosted model and does not use an OpenAI account balance.

The installed version is recorded immediately after installation and will later
be pinned in the environment specification for reproducibility.

In [25]:
# Install the OpenAI-compatible client into the Python environment used by
# the current Jupyter kernel. This modifies the local environment but does
# not make an API request and does not consume gateway quota.
%pip install openai

Note: you may need to restart the kernel to use updated packages.


### Probe basic UoA Gateway connectivity

This probe sends one minimal, non-research request to the UoA Agentic Gateway.
Its purpose is limited to confirming authentication, endpoint connectivity,
model availability and basic Chat Completions compatibility.

The probe contains no network-flow record, sample identifier or ground-truth
label. Automatic retries are disabled so that executing this cell produces at
most one external request. Structured output and inference-control parameters
are tested separately after basic connectivity has been established.

In [26]:
from datetime import datetime, timezone

from openai import OpenAI


# Lock the endpoint and model advertised by the UoA Agentic Gateway.
# The API key is read from the environment and is never printed or saved.
UOA_GATEWAY_BASE_URL = "https://agent.elliottwen.info/v1"
UOA_GATEWAY_MODEL = "MiniMax-M3"

# max_retries=0 ensures that this cell makes at most one external request.
# A finite timeout prevents the Notebook from waiting indefinitely.
uoa_gateway_client = OpenAI(
    api_key=os.environ["UOA_API_KEY"],
    base_url=UOA_GATEWAY_BASE_URL,
    timeout=60.0,
    max_retries=0,
)

# Record when the capability probe was attempted for reproducibility.
probe_started_at_utc = datetime.now(timezone.utc).isoformat()

try:
    # This prompt contains no research data. It tests only the minimum
    # Chat Completions request accepted by the advertised model.
    connectivity_response = uoa_gateway_client.chat.completions.create(
        model=UOA_GATEWAY_MODEL,
        messages=[
            {
                "role": "user",
                "content": "Return exactly the text GATEWAY_OK and nothing else.",
            }
        ],
        max_tokens=16,
    )

    connectivity_choice = connectivity_response.choices[0]
    connectivity_content = (
        connectivity_choice.message.content or ""
    ).strip()

    # Usage fields are accessed defensively because an OpenAI-compatible
    # gateway is not guaranteed to return every optional metadata field.
    connectivity_usage = connectivity_response.usage

    gateway_connectivity_probe = pd.Series(
        {
            "probe_started_at_utc": probe_started_at_utc,
            "request_succeeded": True,
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": getattr(
                connectivity_response,
                "model",
                None,
            ),
            "response_id": getattr(
                connectivity_response,
                "id",
                None,
            ),
            "finish_reason": getattr(
                connectivity_choice,
                "finish_reason",
                None,
            ),
            "response_text": connectivity_content,
            "exact_text_match": connectivity_content == "GATEWAY_OK",
            "prompt_tokens": getattr(
                connectivity_usage,
                "prompt_tokens",
                None,
            ),
            "completion_tokens": getattr(
                connectivity_usage,
                "completion_tokens",
                None,
            ),
            "total_tokens": getattr(
                connectivity_usage,
                "total_tokens",
                None,
            ),
            "automatic_retries": 0,
        },
        name="value",
    )

except Exception as exc:
    # Record a concise diagnostic without exposing the API key.
    # Do not automatically retry: each attempt must remain deliberate.
    gateway_connectivity_probe = pd.Series(
        {
            "probe_started_at_utc": probe_started_at_utc,
            "request_succeeded": False,
            "requested_model": UOA_GATEWAY_MODEL,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "automatic_retries": 0,
        },
        name="value",
    )

display(gateway_connectivity_probe)

probe_started_at_utc    2026-09-01T09:13:15.085128+00:00
request_succeeded                                   True
requested_model                               MiniMax-M3
reported_model                                MiniMax-M3
response_id                    chatcmpl-966008bf2b7627fe
finish_reason                                     length
response_text                                           
exact_text_match                                   False
prompt_tokens                                         64
completion_tokens                                     16
total_tokens                                          80
automatic_retries                                      0
Name: value, dtype: object

### Inspect the returned message fields

The basic request reached the advertised model, but the completion stopped at
the configured token limit before producing visible response content. This
local inspection determines whether the gateway returned additional
model-specific fields, such as reasoning content.

No new network request is made. Only field names, data types, character counts
and bounded previews are displayed.

In [27]:
def summarise_response_field(field_name, field_value, preview_limit=120):
    """
    Produce a bounded summary of one returned message field.

    Parameters
    ----------
    field_name : str
        Name of the field returned by the API client.
    field_value : object
        Parsed value stored in that field.
    preview_limit : int, default=120
        Maximum number of characters included in the preview.

    Returns
    -------
    dict
        Field name, Python type, approximate character count and a bounded
        preview. The full response value is not printed.

    Notes
    -----
    This function operates only on the response already held in memory.
    It makes no external request.
    """
    if isinstance(field_value, str):
        field_preview = field_value[:preview_limit]
        character_count = len(field_value)
    else:
        field_representation = repr(field_value)
        field_preview = field_representation[:preview_limit]
        character_count = len(field_representation)

    return {
        "field_name": field_name,
        "python_type": type(field_value).__name__,
        "character_count": character_count,
        "preview": field_preview,
    }


# Convert the already-returned message into a local dictionary.
# exclude_none=True removes fields that the gateway did not populate.
returned_message_fields = (
    connectivity_choice.message.model_dump(exclude_none=True)
)

# Summarise every populated response field without printing unbounded content.
returned_message_field_summary = pd.DataFrame(
    [
        summarise_response_field(field_name, field_value)
        for field_name, field_value in returned_message_fields.items()
    ]
)

display(returned_message_field_summary)

,field_name,python_type,character_count,preview
0,role,str,9,assistant
1,reasoning_content,str,63,"We need to respond to user: ""Return exactly th..."
2,provider_specific_fields,dict,97,"{'reasoning': 'We need to respond to user: ""Re..."


### Confirm visible completion with a bounded output budget

The first connectivity request authenticated successfully but exhausted its
16-token completion budget during the model's reasoning phase. This second
non-research probe increases the maximum completion budget to 128 tokens while
leaving all other request components unchanged.

The purpose is to confirm that the gateway can return visible assistant content
and terminate normally. No research record, sample identifier or ground-truth
label is included, and automatic retries remain disabled.

In [28]:
# Record the second probe time independently from the first request.
visible_completion_probe_started_at_utc = (
    datetime.now(timezone.utc).isoformat()
)

try:
    # Keep the instruction identical to the first probe.
    # Only the maximum completion budget changes from 16 to 128 tokens.
    visible_completion_response = (
        uoa_gateway_client.chat.completions.create(
            model=UOA_GATEWAY_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Return exactly the text GATEWAY_OK "
                        "and nothing else."
                    ),
                }
            ],
            max_tokens=128,
        )
    )

    visible_completion_choice = visible_completion_response.choices[0]
    visible_completion_message = visible_completion_choice.message

    # The standard visible answer is stored in `content`.
    visible_completion_content = (
        visible_completion_message.content or ""
    ).strip()

    # MiniMax reasoning may be returned as a provider-specific extension.
    # We record only its character count, not the reasoning text itself.
    visible_reasoning_content = getattr(
        visible_completion_message,
        "reasoning_content",
        None,
    )
    visible_reasoning_character_count = (
        len(visible_reasoning_content)
        if isinstance(visible_reasoning_content, str)
        else 0
    )

    visible_completion_usage = visible_completion_response.usage

    visible_completion_probe = pd.Series(
        {
            "probe_started_at_utc": (
                visible_completion_probe_started_at_utc
            ),
            "request_succeeded": True,
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": getattr(
                visible_completion_response,
                "model",
                None,
            ),
            "response_id": getattr(
                visible_completion_response,
                "id",
                None,
            ),
            "finish_reason": getattr(
                visible_completion_choice,
                "finish_reason",
                None,
            ),
            "response_text": visible_completion_content,
            "exact_text_match": (
                visible_completion_content == "GATEWAY_OK"
            ),
            "reasoning_character_count": (
                visible_reasoning_character_count
            ),
            "prompt_tokens": getattr(
                visible_completion_usage,
                "prompt_tokens",
                None,
            ),
            "completion_tokens": getattr(
                visible_completion_usage,
                "completion_tokens",
                None,
            ),
            "total_tokens": getattr(
                visible_completion_usage,
                "total_tokens",
                None,
            ),
            "maximum_completion_tokens": 128,
            "automatic_retries": 0,
        },
        name="value",
    )

except Exception as exc:
    # Preserve a concise failure record without exposing credentials.
    visible_completion_probe = pd.Series(
        {
            "probe_started_at_utc": (
                visible_completion_probe_started_at_utc
            ),
            "request_succeeded": False,
            "requested_model": UOA_GATEWAY_MODEL,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "maximum_completion_tokens": 128,
            "automatic_retries": 0,
        },
        name="value",
    )

display(visible_completion_probe)

probe_started_at_utc         2026-09-01T09:13:18.797563+00:00
request_succeeded                                        True
requested_model                                    MiniMax-M3
reported_model                                     MiniMax-M3
response_id                         chatcmpl-8d0df81fbcda1116
finish_reason                                            stop
response_text                                      GATEWAY_OK
exact_text_match                                         True
reasoning_character_count                                 124
prompt_tokens                                              64
completion_tokens                                          39
total_tokens                                              103
maximum_completion_tokens                                 128
automatic_retries                                           0
Name: value, dtype: object

### Basic-connectivity finding

The UoA Agentic Gateway authenticated successfully and returned the advertised
`MiniMax-M3` model identifier, a response ID, finish status and token-usage
metadata.

The initial 16-token probe terminated with `finish_reason="length"` before
visible content was produced. Local inspection showed that the available
completion budget had been consumed by provider-specific reasoning content.
When the maximum completion budget was increased to 128 tokens, the model
returned exactly `GATEWAY_OK` and terminated normally with
`finish_reason="stop"`.

The two probes consumed 80 and 150 total tokens respectively. Their combined
230-token usage and two successful requests matched the independent UoA Gateway
usage dashboard.

These results establish authentication, endpoint connectivity, advertised-model
availability, basic Chat Completions compatibility and usage-accounting
consistency. They do not yet establish support for deterministic inference
parameters, strict JSON Schema enforcement or the planned research response
format.

### Probe acceptance of zero-temperature inference

This capability probe repeats the successful non-research request while adding
only `temperature=0.0`. Its purpose is to determine whether the UoA Gateway and
the advertised model accept the intended low-randomness setting.

Successful acceptance of this parameter does not prove bit-for-bit
determinism. It establishes only that the request is accepted and produces a
normal visible completion under the requested setting. No research data is
included, automatic retries remain disabled, and at most one request is made.

In [29]:
# Record the attempt time so that gateway capabilities can be associated
# with the date on which they were empirically tested.
temperature_probe_started_at_utc = (
    datetime.now(timezone.utc).isoformat()
)

try:
    # This request is identical to the successful visible-completion probe,
    # except that temperature=0.0 is now supplied.
    temperature_probe_response = (
        uoa_gateway_client.chat.completions.create(
            model=UOA_GATEWAY_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Return exactly the text GATEWAY_OK "
                        "and nothing else."
                    ),
                }
            ],
            temperature=0.0,
            max_tokens=128,
        )
    )

    temperature_probe_choice = temperature_probe_response.choices[0]
    temperature_probe_content = (
        temperature_probe_choice.message.content or ""
    ).strip()
    temperature_probe_usage = temperature_probe_response.usage

    temperature_capability_probe = pd.Series(
        {
            "probe_started_at_utc": temperature_probe_started_at_utc,
            "request_succeeded": True,
            "temperature_requested": 0.0,
            "temperature_parameter_accepted": True,
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": getattr(
                temperature_probe_response,
                "model",
                None,
            ),
            "finish_reason": getattr(
                temperature_probe_choice,
                "finish_reason",
                None,
            ),
            "response_text": temperature_probe_content,
            "exact_text_match": (
                temperature_probe_content == "GATEWAY_OK"
            ),
            "prompt_tokens": getattr(
                temperature_probe_usage,
                "prompt_tokens",
                None,
            ),
            "completion_tokens": getattr(
                temperature_probe_usage,
                "completion_tokens",
                None,
            ),
            "total_tokens": getattr(
                temperature_probe_usage,
                "total_tokens",
                None,
            ),
            "maximum_completion_tokens": 128,
            "automatic_retries": 0,
        },
        name="value",
    )

except Exception as exc:
    # A rejected request is evidence that this gateway/model combination
    # does not accept the requested parameter through this API route.
    temperature_capability_probe = pd.Series(
        {
            "probe_started_at_utc": temperature_probe_started_at_utc,
            "request_succeeded": False,
            "temperature_requested": 0.0,
            "temperature_parameter_accepted": False,
            "requested_model": UOA_GATEWAY_MODEL,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "maximum_completion_tokens": 128,
            "automatic_retries": 0,
        },
        name="value",
    )

display(temperature_capability_probe)

probe_started_at_utc              2026-09-01T09:13:22.355550+00:00
request_succeeded                                             True
temperature_requested                                          0.0
temperature_parameter_accepted                                True
requested_model                                         MiniMax-M3
reported_model                                          MiniMax-M3
finish_reason                                                 stop
response_text                                           GATEWAY_OK
exact_text_match                                              True
prompt_tokens                                                   64
completion_tokens                                               40
total_tokens                                                   104
maximum_completion_tokens                                      128
automatic_retries                                                0
Name: value, dtype: object

### Zero-temperature capability finding

The UoA Gateway accepted `temperature=0.0` for the advertised `MiniMax-M3`
model. The request completed normally, returned exactly `GATEWAY_OK` and
reported token usage without an automatic retry.

This result demonstrates API-level acceptance of the requested temperature
parameter. It does not establish that the gateway or underlying model honours
the parameter exactly, nor does it establish bit-for-bit deterministic
inference. The zero-temperature setting is therefore treated as a
low-randomness control rather than a guarantee of identical repeated outputs.

### Probe acceptance of a fixed random seed

This capability probe adds `seed=742` to the previously successful
zero-temperature request. The value is an arbitrary but documented experimental
constant chosen for traceability to COMPSCI 742.

The probe tests whether the gateway accepts the seed parameter and whether the
request completes normally. Parameter acceptance does not prove that the
gateway or underlying model uses the seed, and it does not guarantee identical
outputs across repeated requests or future service versions.

No research data is included, automatic retries remain disabled, and at most
one request is made.

In [30]:
# Use one documented seed value across both representation conditions if the
# UoA Gateway accepts this parameter.
CANDIDATE_INFERENCE_SEED = 742

seed_probe_started_at_utc = datetime.now(timezone.utc).isoformat()

try:
    # Relative to the successful temperature probe, only seed=742 is added.
    seed_probe_response = (
        uoa_gateway_client.chat.completions.create(
            model=UOA_GATEWAY_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Return exactly the text GATEWAY_OK "
                        "and nothing else."
                    ),
                }
            ],
            temperature=0.0,
            seed=CANDIDATE_INFERENCE_SEED,
            max_tokens=128,
        )
    )

    seed_probe_choice = seed_probe_response.choices[0]
    seed_probe_content = (
        seed_probe_choice.message.content or ""
    ).strip()
    seed_probe_usage = seed_probe_response.usage

    seed_capability_probe = pd.Series(
        {
            "probe_started_at_utc": seed_probe_started_at_utc,
            "request_succeeded": True,
            "temperature_requested": 0.0,
            "seed_requested": CANDIDATE_INFERENCE_SEED,
            "seed_parameter_accepted": True,
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": getattr(
                seed_probe_response,
                "model",
                None,
            ),
            "system_fingerprint": getattr(
                seed_probe_response,
                "system_fingerprint",
                None,
            ),
            "finish_reason": getattr(
                seed_probe_choice,
                "finish_reason",
                None,
            ),
            "response_text": seed_probe_content,
            "exact_text_match": (
                seed_probe_content == "GATEWAY_OK"
            ),
            "prompt_tokens": getattr(
                seed_probe_usage,
                "prompt_tokens",
                None,
            ),
            "completion_tokens": getattr(
                seed_probe_usage,
                "completion_tokens",
                None,
            ),
            "total_tokens": getattr(
                seed_probe_usage,
                "total_tokens",
                None,
            ),
            "maximum_completion_tokens": 128,
            "automatic_retries": 0,
        },
        name="value",
    )

except Exception as exc:
    # A rejected request indicates that seed cannot be used through this
    # gateway/model/API combination in its current form.
    seed_capability_probe = pd.Series(
        {
            "probe_started_at_utc": seed_probe_started_at_utc,
            "request_succeeded": False,
            "temperature_requested": 0.0,
            "seed_requested": CANDIDATE_INFERENCE_SEED,
            "seed_parameter_accepted": False,
            "requested_model": UOA_GATEWAY_MODEL,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "maximum_completion_tokens": 128,
            "automatic_retries": 0,
        },
        name="value",
    )

display(seed_capability_probe)

probe_started_at_utc         2026-09-01T09:13:25.897841+00:00
request_succeeded                                        True
temperature_requested                                     0.0
seed_requested                                            742
seed_parameter_accepted                                  True
requested_model                                    MiniMax-M3
reported_model                                     MiniMax-M3
system_fingerprint                       vllm-0.27.1-c607c424
finish_reason                                            stop
response_text                                      GATEWAY_OK
exact_text_match                                         True
prompt_tokens                                              64
completion_tokens                                          40
total_tokens                                              104
maximum_completion_tokens                                 128
automatic_retries                                           0
Name: va

### Probe JSON Schema response compatibility

The experiment requires machine-parseable responses with a fixed structure.
This non-research capability probe tests whether the UoA Gateway accepts an
OpenAI-compatible `json_schema` response format and whether the returned visible
content conforms to a minimal predefined schema.

The probe uses a deliberately small schema unrelated to the network-flow task.
It retains the candidate inference controls `temperature=0.0` and `seed=742`.
No research record, sample identifier or ground-truth label is included, and
automatic retries remain disabled.

A conforming response demonstrates request acceptance and observed schema
compliance. A single successful response does not prove that the provider will
enforce the schema correctly for every possible request.

In [31]:
# Define a small, provider-neutral schema for capability testing.
# The real network-flow response schema is not used until this basic API
# capability has been established.
MINIMAL_SCHEMA_PROBE = {
    "type": "object",
    "properties": {
        "status": {
            "type": "string",
            "const": "ok",
        },
        "code": {
            "type": "integer",
            "const": 742,
        },
    },
    "required": [
        "status",
        "code",
    ],
    "additionalProperties": False,
}

schema_probe_started_at_utc = datetime.now(timezone.utc).isoformat()

try:
    # Ask the gateway to constrain the visible response to the predefined
    # JSON Schema. The output allowance includes MiniMax's reasoning tokens.
    schema_probe_response = (
        uoa_gateway_client.chat.completions.create(
            model=UOA_GATEWAY_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": (
                        "Return an object indicating that the gateway "
                        "schema probe succeeded, using status 'ok' and "
                        "code 742."
                    ),
                }
            ],
            temperature=0.0,
            seed=CANDIDATE_INFERENCE_SEED,
            max_tokens=512,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "uoa_gateway_schema_probe",
                    "strict": True,
                    "schema": MINIMAL_SCHEMA_PROBE,
                },
            },
        )
    )

    schema_probe_choice = schema_probe_response.choices[0]
    schema_probe_message = schema_probe_choice.message
    schema_probe_content = (
        schema_probe_message.content or ""
    ).strip()

    # Parse the visible response locally. A successful API request alone
    # does not guarantee that the returned text is valid JSON.
    try:
        schema_probe_parsed = json.loads(schema_probe_content)
        schema_probe_json_valid = True
        schema_probe_json_error = None
    except json.JSONDecodeError as exc:
        schema_probe_parsed = None
        schema_probe_json_valid = False
        schema_probe_json_error = str(exc)

    # For this deliberately minimal schema, exact object equality checks
    # every required field, value and the absence of extra fields.
    schema_probe_exact_match = (
        schema_probe_parsed
        == {
            "status": "ok",
            "code": 742,
        }
    )

    schema_probe_reasoning = getattr(
        schema_probe_message,
        "reasoning_content",
        None,
    )
    schema_probe_usage = schema_probe_response.usage

    json_schema_capability_probe = pd.Series(
        {
            "probe_started_at_utc": schema_probe_started_at_utc,
            "request_succeeded": True,
            "json_schema_parameter_accepted": True,
            "temperature_requested": 0.0,
            "seed_requested": CANDIDATE_INFERENCE_SEED,
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": getattr(
                schema_probe_response,
                "model",
                None,
            ),
            "system_fingerprint": getattr(
                schema_probe_response,
                "system_fingerprint",
                None,
            ),
            "finish_reason": getattr(
                schema_probe_choice,
                "finish_reason",
                None,
            ),
            "response_text": schema_probe_content,
            "json_valid": schema_probe_json_valid,
            "schema_probe_exact_match": schema_probe_exact_match,
            "json_error": schema_probe_json_error,
            "reasoning_character_count": (
                len(schema_probe_reasoning)
                if isinstance(schema_probe_reasoning, str)
                else 0
            ),
            "prompt_tokens": getattr(
                schema_probe_usage,
                "prompt_tokens",
                None,
            ),
            "completion_tokens": getattr(
                schema_probe_usage,
                "completion_tokens",
                None,
            ),
            "total_tokens": getattr(
                schema_probe_usage,
                "total_tokens",
                None,
            ),
            "maximum_completion_tokens": 512,
            "automatic_retries": 0,
        },
        name="value",
    )

except Exception as exc:
    # Record rejection or incompatibility without exposing the API key.
    json_schema_capability_probe = pd.Series(
        {
            "probe_started_at_utc": schema_probe_started_at_utc,
            "request_succeeded": False,
            "json_schema_parameter_accepted": False,
            "temperature_requested": 0.0,
            "seed_requested": CANDIDATE_INFERENCE_SEED,
            "requested_model": UOA_GATEWAY_MODEL,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "maximum_completion_tokens": 512,
            "automatic_retries": 0,
        },
        name="value",
    )

display(json_schema_capability_probe)

probe_started_at_utc                    2026-09-01T09:13:29.451549+00:00
request_succeeded                                                   True
json_schema_parameter_accepted                                      True
temperature_requested                                                0.0
seed_requested                                                       742
requested_model                                               MiniMax-M3
reported_model                                                MiniMax-M3
system_fingerprint                                  vllm-0.27.1-c607c424
finish_reason                                                       stop
response_text                     {\n  "status": "ok",\n  "code": 742\n}
json_valid                                                          True
schema_probe_exact_match                                            True
json_error                                                          None
reasoning_character_count                          

### JSON Schema capability finding

The UoA Gateway accepted the OpenAI-compatible `json_schema` response-format
parameter together with `temperature=0.0` and `seed=742`. The advertised
`MiniMax-M3` model returned valid JSON that matched the complete minimal probe
object, and the response terminated normally without an automatic retry.

The minimal response consumed 507 of the available 512 completion tokens,
including provider-specific reasoning content. This indicates that a
512-token completion allowance is not sufficiently conservative for the
planned research response containing one classification, five feature
citations and five rationales.

A larger bounded completion allowance will therefore be evaluated during the
paired research smoke test. The successful probe demonstrates observed schema
compliance for this request; it does not prove universal provider-side schema
enforcement for every possible input.

### Run the first paired research smoke test

This smoke test applies the complete experimental request contract to the two
information-equivalent representations of `pilot_001`. The structured and
deterministic-text requests use the same model, system instruction, output
schema, temperature, seed, completion limit and API client. Only the validated
record representation differs.

The outer sample identifier is retained locally for pairing but is not included
in either request sent to the LLM. The private ground-truth label is neither
loaded nor inspected at this stage.

Exactly two external requests are made: one for each representation condition.
The results are used only to test execution, output sufficiency and mechanical
validation. They are not used to estimate detection performance or tune the
prompt against the true label.

In [32]:
import time


# Candidate completion allowance for the full research response.
# This limit includes both provider-specific reasoning tokens and visible output.
PAIRED_SMOKE_MAX_TOKENS = 4096

# Prevent accidental duplicate execution within the current kernel session.
# Intentional reruns should first delete this variable after documenting why.
if "paired_smoke_test_results" in globals():
    raise RuntimeError(
        "The paired smoke test has already been executed in this kernel. "
        "Do not repeat paid or quota-consuming requests accidentally."
    )


def run_uoa_research_request(
    request_contract,
    condition,
    local_sample_id,
):
    """
    Execute one provider-specific LLM request from a validated logical contract.

    Parameters
    ----------
    request_contract : dict
        Provider-independent contract containing the common system instruction,
        one representation-specific user message and the common output schema.
    condition : str
        Local representation label, either ``structured`` or
        ``deterministic_text``. This value is recorded locally and is not sent
        to the model.
    local_sample_id : str
        Local identifier used to pair experimental results. This value is not
        included in the API messages.

    Returns
    -------
    dict
        Local execution record containing request metadata, visible response,
        token usage and a bounded error description when applicable.

    Notes
    -----
    The UoA client has ``max_retries=0``. Each function call therefore makes at
    most one external request. Provider-specific reasoning text is not retained;
    only its character count is recorded.
    """
    request_started_at_utc = datetime.now(timezone.utc).isoformat()
    request_start_time = time.perf_counter()

    try:
        response = uoa_gateway_client.chat.completions.create(
            model=UOA_GATEWAY_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": request_contract["system_instruction"],
                },
                {
                    "role": "user",
                    "content": request_contract["user_message"],
                },
            ],
            temperature=0.0,
            seed=CANDIDATE_INFERENCE_SEED,
            max_tokens=PAIRED_SMOKE_MAX_TOKENS,
            response_format={
                "type": "json_schema",
                "json_schema": request_contract["output_schema"],
            },
        )

        elapsed_seconds = time.perf_counter() - request_start_time
        response_choice = response.choices[0]
        response_message = response_choice.message
        visible_content = (response_message.content or "").strip()

        reasoning_content = getattr(
            response_message,
            "reasoning_content",
            None,
        )
        usage = response.usage

        return {
            # Local pairing metadata: never included in the LLM messages.
            "sample_id": local_sample_id,
            "condition": condition,

            # Execution and provider metadata.
            "request_started_at_utc": request_started_at_utc,
            "request_succeeded": True,
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": getattr(response, "model", None),
            "system_fingerprint": getattr(
                response,
                "system_fingerprint",
                None,
            ),
            "response_id": getattr(response, "id", None),
            "finish_reason": getattr(
                response_choice,
                "finish_reason",
                None,
            ),
            "elapsed_seconds": elapsed_seconds,

            # The visible response is the only model explanation evaluated.
            "response_text": visible_content,
            "visible_response_characters": len(visible_content),

            # Hidden reasoning is not retained or analysed.
            "reasoning_character_count": (
                len(reasoning_content)
                if isinstance(reasoning_content, str)
                else 0
            ),

            # Usage metadata supports quota and cost auditing.
            "prompt_tokens": getattr(
                usage,
                "prompt_tokens",
                None,
            ),
            "completion_tokens": getattr(
                usage,
                "completion_tokens",
                None,
            ),
            "total_tokens": getattr(
                usage,
                "total_tokens",
                None,
            ),

            # Locked candidate inference controls.
            "temperature": 0.0,
            "seed": CANDIDATE_INFERENCE_SEED,
            "maximum_completion_tokens": PAIRED_SMOKE_MAX_TOKENS,
            "automatic_retries": 0,
            "error_type": None,
            "error_message": None,
        }

    except Exception as exc:
        elapsed_seconds = time.perf_counter() - request_start_time

        # Preserve the failure as experimental metadata rather than silently
        # retrying or discarding it.
        return {
            "sample_id": local_sample_id,
            "condition": condition,
            "request_started_at_utc": request_started_at_utc,
            "request_succeeded": False,
            "requested_model": UOA_GATEWAY_MODEL,
            "reported_model": None,
            "system_fingerprint": None,
            "response_id": None,
            "finish_reason": None,
            "elapsed_seconds": elapsed_seconds,
            "response_text": None,
            "visible_response_characters": 0,
            "reasoning_character_count": 0,
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "temperature": 0.0,
            "seed": CANDIDATE_INFERENCE_SEED,
            "maximum_completion_tokens": PAIRED_SMOKE_MAX_TOKENS,
            "automatic_retries": 0,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }


# Confirm locally that both outer records belong to the same pilot sample.
smoke_sample_id = structured_records[0]["sample_id"]

assert smoke_sample_id == text_records[0]["sample_id"], (
    "The selected structured and text records are not a valid pair."
)

# Execute exactly one request per representation condition.
paired_smoke_test_results = [
    run_uoa_research_request(
        request_contract=first_structured_request_contract,
        condition="structured",
        local_sample_id=smoke_sample_id,
    ),
    run_uoa_research_request(
        request_contract=first_text_request_contract,
        condition="deterministic_text",
        local_sample_id=smoke_sample_id,
    ),
]

# Display a bounded operational summary. The full visible responses are
# inspected and validated separately below.
paired_smoke_test_summary = pd.DataFrame(
    [
        {
            key: value
            for key, value in result.items()
            if key != "response_text"
        }
        for result in paired_smoke_test_results
    ]
)

display(paired_smoke_test_summary)

,sample_id,condition,request_started_at_utc,request_succeeded,requested_model,reported_model,system_fingerprint,response_id,finish_reason,elapsed_seconds,...,reasoning_character_count,prompt_tokens,completion_tokens,total_tokens,temperature,seed,maximum_completion_tokens,automatic_retries,error_type,error_message
0,pilot_001,structured,2026-09-01T09:13:37.738848+00:00,True,MiniMax-M3,MiniMax-M3,vllm-0.27.1-c607c424,chatcmpl-bd65c6a343bab456,length,43.823200,...,5127,1380,4096,5476,0.0,742,4096,0,None,None
1,pilot_001,deterministic_text,2026-09-01T09:14:21.562350+00:00,True,MiniMax-M3,MiniMax-M3,vllm-0.27.1-c607c424,chatcmpl-9b67fd48154aae1c,length,43.850676,...,12351,1358,4096,5454,0.0,742,4096,0,None,None


In [ ]:
# Retry the same paired smoke-test record with a larger completion budget.
#
# Research purpose
# ----------------
# The first paired attempt reached the 4,096-token ceiling under both
# representation conditions. This retry changes only the completion-token
# ceiling. The model, prompt, schema, temperature, seed and underlying sample
# remain unchanged.
#
# The request order is reversed in this retry so that the structured condition
# is not always submitted first.
#
# Expected output
# ---------------
# Two rows:
#   1. deterministic_text
#   2. structured
#
# A usable response should normally have:
#   request_succeeded == True
#   finish_reason == "stop"
#   completion_tokens < 8192
#   error_type == None
#
# This cell makes exactly two external API requests.

PAIRED_SMOKE_RETRY_MAX_TOKENS = 8192

# Prevent an accidental rerun from silently consuming another two requests.
if "paired_smoke_retry_results" in globals():
    raise RuntimeError(
        "paired_smoke_retry_results already exists. "
        "Do not rerun this cell unless the previous retry is deliberately removed."
    )

# The existing request function reads this global completion-token setting.
# Only this ceiling changes relative to the first attempt.
PAIRED_SMOKE_MAX_TOKENS = PAIRED_SMOKE_RETRY_MAX_TOKENS

# Confirm that both representations still refer to exactly the same sample.
retry_sample_id = structured_records[0]["sample_id"]

assert retry_sample_id == text_records[0]["sample_id"], (
    "The structured and deterministic-text records do not belong to the "
    "same sample."
)

# Reverse the order used in the first attempt:
# deterministic text first, then structured values.
paired_smoke_retry_results = [
    run_uoa_research_request(
        request_contract=first_text_request_contract,
        condition="deterministic_text",
        local_sample_id=retry_sample_id,
    ),
    run_uoa_research_request(
        request_contract=first_structured_request_contract,
        condition="structured",
        local_sample_id=retry_sample_id,
    ),
]

# Display only the operational fields needed to decide whether the retry
# produced complete visible responses. Full response text remains available
# inside paired_smoke_retry_results for validation in the next step.
retry_summary_columns = [
    "sample_id",
    "condition",
    "request_succeeded",
    "requested_model",
    "reported_model",
    "system_fingerprint",
    "finish_reason",
    "elapsed_seconds",
    "response_character_count",
    "reasoning_character_count",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "maximum_completion_tokens",
    "automatic_retries",
    "error_type",
    "error_message",
]

paired_smoke_retry_summary = pd.DataFrame(
    paired_smoke_retry_results
)[retry_summary_columns]

display(paired_smoke_retry_summary)

In [34]:
# Repair only the local display step.
#
# No API request is made here. The two retry responses already stored in
# paired_smoke_retry_results are preserved and reused.

paired_smoke_retry_summary = pd.DataFrame(paired_smoke_retry_results).copy()

# Calculate the visible-response length directly from the stored response text,
# because run_uoa_research_request() does not create a field named
# "response_character_count".
paired_smoke_retry_summary["visible_response_character_count"] = (
    paired_smoke_retry_summary["response_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

# Select only columns that actually exist. This makes the diagnostic display
# robust to optional provider metadata.
desired_retry_columns = [
    "sample_id",
    "condition",
    "request_succeeded",
    "requested_model",
    "reported_model",
    "system_fingerprint",
    "finish_reason",
    "elapsed_seconds",
    "visible_response_character_count",
    "reasoning_character_count",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "maximum_completion_tokens",
    "automatic_retries",
    "error_type",
    "error_message",
]

available_retry_columns = [
    column
    for column in desired_retry_columns
    if column in paired_smoke_retry_summary.columns
]

display(paired_smoke_retry_summary[available_retry_columns])

,sample_id,condition,request_succeeded,requested_model,reported_model,system_fingerprint,finish_reason,elapsed_seconds,visible_response_character_count,reasoning_character_count,prompt_tokens,completion_tokens,total_tokens,maximum_completion_tokens,automatic_retries,error_type,error_message
0,pilot_001,deterministic_text,False,MiniMax-M3,None,None,None,60.010961,0,0,None,None,None,8192,0,APITimeoutError,Request timed out.
1,pilot_001,structured,False,MiniMax-M3,None,None,None,60.228113,0,0,None,None,None,8192,0,APITimeoutError,Request timed out.


## Pilot inference conclusion and blocking gate

The UoA Agentic Gateway was successfully configured through the
`UOA_API_KEY` environment variable and accessed using its OpenAI-compatible
interface. The requested and reported model was `MiniMax-M3`. Connectivity,
`temperature=0.0`, `seed=742` and strict JSON-Schema response formatting were
successfully accepted during the capability probes.

A paired smoke test was then performed for `pilot_001`, using the same
underlying 46-feature network-flow record under the structured and
deterministic-text conditions. The true class label and local sample identifier
were not included in either model request.

### Observed execution behaviour

- With a maximum completion budget of 4,096 tokens, both requests reached the
  full completion-token ceiling and returned `finish_reason="length"`.
- The responses were therefore treated as truncated and were not used for
  detection-performance or explanation-quality evaluation.
- A controlled retry changed only the completion-token ceiling from 4,096 to
  8,192 tokens and reversed the condition order.
- Both retry requests reached the client's approximately 60-second timeout and
  returned `APITimeoutError` without a visible response.
- Automatic retries remained disabled, preventing duplicate unrecorded
  requests.

### Scientific interpretation

These observations are infrastructure and protocol findings, not evidence for
RQ1 or RQ2. Because both representation conditions encountered the same
completion-budget and timeout constraints, this smoke test does not establish
a representation effect.

The pilot confirms that the saved provider-independent request protocol can be
translated to the UoA Gateway, but the current provider-specific execution
configuration is not yet suitable for the full 400-request experiment.
No detection metrics, feature-grounding measures or attribution-agreement
measures will be calculated from these incomplete responses.

### Blocking decision

Full LLM inference is blocked until a subsequent implementation step:

1. establishes an adequate client timeout and completion-token budget, or
2. reduces unnecessary reasoning consumption without changing the
   information supplied to either representation condition;
3. confirms complete, schema-valid and grounded responses for a small paired
   sample; and
4. implements checkpointed, resumable and order-balanced batch execution.

Notebook 05 therefore completes the provider-integration and smoke-test stage,
while correctly withholding the formal experiment pending resolution of the
provider-specific execution constraint.